# Clasificador IoT Multi-Dataset - Taxonomía Granular

## Actualización Reciente: Nueva Estrategia de Clasificación

**Versión**: 2.0 - Taxonomía Granular (17 categorías específicas)

### Cambios Principales

**Estrategia anterior** (6 clases genéricas):
- Camera, Audio, Sensor, Hub, Switch, Peripheral

**Nueva estrategia** (17 clases específicas):
1. **Asistentes de Voz**: Alexa, GoogleHome, HomePod, SmartSpeaker
2. **Cámaras**: SecurityCamera, IndoorCamera, MonitorCamera
3. **Sensores**: MotionSensor, EnvironmentalSensor, HealthSensor
4. **Control**: SmartBulb, SmartPlug, SmartSwitch
5. **Seguridad**: SmartLock
6. **Otros**: Hub, Printer, Other

### Objetivo

Clasificación específica de dispositivos IoT para:
- Demo con Alexa: Alexa se clasifica específicamente vs otros asistentes
- Mayor precisión: Categorías más granulares por tipo de dispositivo
- Sin entrenamiento cruzado: Enfoque directo en clasificación específica

### Datos Actuales

**Datasets habilitados**: UNSW (27 archivos) + IoT-23 (9 archivos) + Deakin (119 archivos)
- **Total**: 153 archivos
- **Muestras**: ~473,000 paquetes (con MAX_PKTS=1000)
- **Clases detectadas**: 12 de 17 categorías definidas

**Clase principal para demo**: **Alexa** (157,401 muestras, 33.4%)

In [ ]:
# Install dependencies in the notebook environment
%pip install -r requirements.txt

In [ ]:
%pip install "tensorflow[and-cuda]"

In [ ]:
import tensorflow as tf
# Check for GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f" GPU Detected: {len(gpus)} device(s). TensorFlow will use it for the CNN.")
        
        # Print details
        for gpu in gpus:
            print(f"  - Name: {gpu.name}, Type: {gpu.device_type}")
            
    except RuntimeError as e:
        print(e)
else:
    print(" No GPU detected. The CNN will train on CPU (slower).")
    print("To enable GPU, ensure you have the NVIDIA drivers and CUDA Toolkit installed.")

In [ ]:
import os
import numpy as np
from scapy.all import rdpcap, Ether, IP, TCP, UDP

# --- 1. CONFIGURACIÓN DE RUTAS Y DATASETS ---
DATA_ROOT = "/media/orb/SSD 1TB2/TMA"  

# Control de activación de datasets (poner False para deshabilitar temporalmente)
DATASET_ENABLED = {
    "UNSW":   True,   # Nombres descriptivos en archivos
    "IoT23":  True,   # Mapeo manual por carpeta
    "Deakin": True,   # Mapeo MAC→Dispositivo en CSV
    "ACI":    False   # Sin mapeo archivo→dispositivo (descartado)
}

PATHS = {
    "UNSW":     os.path.join(DATA_ROOT, "UNSW-IoTraffic/pcaps"),
    "Deakin":   os.path.join(DATA_ROOT, "Deakin_IoT/pcapIoT"),
    "Deakin_CSV": os.path.join(DATA_ROOT, "Deakin_IoT/CSVs/macAddresses.csv"),
    "IoT23":    os.path.join(DATA_ROOT, "IoT-23/Malware-Project/BigDataset/IoTScenarios") 
}

MAX_PKTS = 50000  # Aumentado: 50K paquetes por archivo (antes 10K)
MAX_LEN = 500     # IMPORTANTE: 500-784 bytes para que la CNN vea patrones útiles

# Control de limitación de archivos Deakin - SIN LÍMITES para máximo rendimiento
DEAKIN_MAX_FILES = 999          # SIN LÍMITE: usar todos los archivos Deakin
DEAKIN_MAX_FILES_ALEXA = 999    # SIN LÍMITE: usar todos los archivos Alexa
DEAKIN_ALEXA_KEYWORD = "echo"   # Keyword para identificar archivos Alexa

# --- 2. TAXONOMÍA DE DISPOSITIVOS IoT (ETIQUETAS GRANULARES) ---
# Nueva estrategia: Clasificación específica por dispositivo/función
# Sin entrenamiento cruzado - enfoque en precisión para demo con Alexa

# A) Mapeo Exclusivo para IoT-23 (Basado en carpetas específicas)
IOT23_CLEAN_MAP = {
    "Capture-4-1": "Hub",          # Philips Hue Bridge
    "Capture-5-1": "Alexa",        # Amazon Echo Dot - ESPECÍFICO para demo
    "Capture-7-1": "SmartLock",    # Somfy Door Lock
    "Somfy-":      "SmartLock"     # Subcarpetas Somfy
}

# B) Mapeo Granular Multi-Keyword - 17 categorías específicas
LABEL_MAP = {
    # === ASISTENTES DE VOZ - ESPECÍFICOS POR MARCA ===
    "Alexa": [
        "echo", "alexa", "dot", "show", "amazon_echo", "amazonechodot",
        "amazon echo", "echo dot"
    ],
    
    "GoogleHome": [
        "googlehome", "google_home", "google home", "nest_audio", 
        "nest audio", "google assistant"
    ],
    
    "HomePod": [
        "homepod", "apple homepod", "apple_homepod"
    ],
    
    "SmartSpeaker": [
        "speaker", "sonos", "triby", "barbie"
    ],
    
    # === CÁMARAS - POR TIPO DE USO ===
    "SecurityCamera": [
        "ring", "arlo", "canary", "nest cam", "dropcam", "outdoor", 
        "doorbell", "security", "outdoor cam"
    ],
    
    "IndoorCamera": [
        "baby", "d-link", "simcam", "yi", "indoor", "belkin",
        "tplink cam", "ezviz", "indoor cam", "baby monitor"
    ],
    
    "MonitorCamera": [
        "pixstar", "withings", "samsung cam", "ubiquiti", "monitor cam"
    ],
    
    # === SENSORES - POR FUNCIÓN ===
    "MotionSensor": [
        "motion", "sensor", "aqara", "alarm", "motion sensor"
    ],
    
    "EnvironmentalSensor": [
        "air", "weather", "netatmo", "awair", "quality", "station", 
        "smoke", "co2", "protect", "air quality", "weather station"
    ],
    
    "HealthSensor": [
        "blipcare", "bpmeter", "blood pressure", "withings", "sleep", 
        "scale", "body+", "health"
    ],
    
    # === ILUMINACIÓN Y CONTROL ===
    "SmartBulb": [
        "bulb", "light", "lifx", "hue bulb", "smart light", "lighting"
    ],
    
    "SmartPlug": [
        "plug", "socket", "wemo", "outlet", "ihome", "tplink plug", 
        "topersun", "smart plug"
    ],
    
    "SmartSwitch": [
        "switch", "tuya switch", "smart switch"
    ],
    
    # === SEGURIDAD DE ACCESO ===
    "SmartLock": [
        "lock", "august", "door", "smartdoor", "smart lock", "door lock"
    ],
    
    # === HUBS Y CONTROLADORES ===
    "Hub": [
        "hue bridge", "hub", "smartthings", "gateway", "bridge", 
        "coordinator", "aeotec", "hue"
    ],
    
    # === OTROS ===
    "Printer": [
        "printer", "hp", "print", "envy"
    ],
    
    "Other": [
        "frame", "photo", "monitor", "watch"
    ]
}

# --- 3. FUNCIÓN DE ASIGNACIÓN DE ETIQUETAS ---
def get_label(filename, folder_name=""):
    """
    Asigna etiquetas específicas priorizando mapeo explícito de IoT-23.
    Nueva taxonomía granular con 17 categorías específicas.
    
    Returns:
        str: Etiqueta específica o "Unknown" si no hay coincidencias
    """
    full_path_str = f"{folder_name}/{filename}"
    
    # --- ESTRATEGIA 1: IoT-23 (Búsqueda Exacta por ID de carpeta) ---
    for key, label in IOT23_CLEAN_MAP.items():
        if key in full_path_str:
            return label

    # --- ESTRATEGIA 2: Búsqueda Granular (UNSW, Deakin) ---
    full_str_lower = full_path_str.lower()
    
    # Priorización: Dispositivos específicos primero, luego categorías más amplias
    # Orden de búsqueda para evitar conflictos (más específico a más general)
    priority_order = [
        # Asistentes de voz - PRIMERO los específicos
        "Alexa", "GoogleHome", "HomePod", "SmartSpeaker",
        # Cámaras - específicas por tipo
        "SecurityCamera", "IndoorCamera", "MonitorCamera",
        # Sensores - por función
        "MotionSensor", "EnvironmentalSensor", "HealthSensor",
        # Iluminación y control
        "SmartBulb", "SmartPlug", "SmartSwitch",
        # Seguridad
        "SmartLock",
        # Hubs
        "Hub",
        # Otros
        "Printer", "Other"
    ]
    
    for label in priority_order:
        keywords = LABEL_MAP.get(label, [])
        for k in keywords:
            if k in full_str_lower:
                return label
                
    return "Unknown"

# --- 4. SANITIZACIÓN DE PAQUETES ---
def sanitize_packet(pkt):
    """
    Limpia datos de identidad (IP/MAC) y normaliza a vector de bytes fijo.
    
    Objetivo: Evitar que el modelo aprenda a identificar dispositivos por su IP/MAC
              en lugar de aprender patrones de tráfico reales.
    """
    try:
        # Enmascarar Capa 2 (Ethernet)
        if Ether in pkt:
            pkt[Ether].src = "00:00:00:00:00:00"
            pkt[Ether].dst = "00:00:00:00:00:00"
        
        # Enmascarar Capa 3 (IP)
        if IP in pkt:
            pkt[IP].src = "0.0.0.0"
            pkt[IP].dst = "0.0.0.0"
        
        # Convertir a Vector de Bytes con Padding/Truncamiento
        byte_list = list(bytes(pkt))
        if len(byte_list) > MAX_LEN:
            return byte_list[:MAX_LEN]
        else:
            return byte_list + [0] * (MAX_LEN - len(byte_list))
    except:
        return None

# --- 4.5 FUNCIONES ESPECÍFICAS PARA DEAKIN ---
def load_deakin_mac_mapping(csv_path):
    """
    Carga el mapeo MAC address → Device Name desde el CSV de Deakin.
    
    Returns:
        dict: Mapeo {mac_address: device_name}
    """
    import csv
    mac_map = {}
    try:
        with open(csv_path, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                mac = row['MAC Address'].lower().strip()
                device = row['Device Name'].strip()
                mac_map[mac] = device
        print(f"   Cargados {len(mac_map)} dispositivos desde {os.path.basename(csv_path)}")
    except Exception as e:
        print(f"   Error cargando MAC mapping: {e}")
    return mac_map

def get_deakin_label(pkt, mac_mapping):
    """
    Extrae la MAC address del paquete y la mapea a una etiqueta genérica.
    Soporta tanto Ethernet como Linux Cooked Capture (SLL).
    
    Args:
        pkt: Paquete Scapy
        mac_mapping: Diccionario {mac: device_name}
        
    Returns:
        str: Etiqueta genérica o "Unknown"
    """
    try:
        # Intentar con Ethernet primero
        if Ether in pkt:
            src_mac = pkt[Ether].src.lower()
            if src_mac in mac_mapping:
                device_name = mac_mapping[src_mac]
                return get_label(device_name, "")
            
            dst_mac = pkt[Ether].dst.lower()
            if dst_mac in mac_mapping:
                device_name = mac_mapping[dst_mac]
                return get_label(device_name, "")
        
        # Si no es Ethernet, intentar con Linux Cooked Capture (SLL)
        # Scapy detecta esto como layer "cooked linux" o CookedLinux
        if hasattr(pkt, 'src') and isinstance(pkt.src, bytes):
            # Convertir bytes MAC a formato string (aa:bb:cc:dd:ee:ff)
            mac_bytes = pkt.src
            if len(mac_bytes) >= 6:
                mac_str = ':'.join(f'{b:02x}' for b in mac_bytes[:6])
                if mac_str in mac_mapping:
                    device_name = mac_mapping[mac_str]
                    return get_label(device_name, "")
        
    except Exception as e:
        pass
    return "Unknown"

# --- 5. CARGADOR DE DATASETS ---
def load_datasets(discovery_mode=True, verbose=True):
    """
    Carga y etiqueta múltiples datasets de tráfico IoT.
    
    Args:
        discovery_mode (bool): Si True, solo imprime estadísticas sin cargar datos
        verbose (bool): Controla nivel de detalle en salida
        
    Returns:
        tuple: (X, y) arrays de NumPy con datos y etiquetas, o (None, None) en modo discovery
    """
    X, y = [], []
    
    print("=" * 70)
    print(f"{'MODO DESCUBRIMIENTO' if discovery_mode else 'CARGANDO DATOS COMPLETOS'}")
    print("=" * 70)
    
    dataset_stats = {}
    
    # Cargar mapeo MAC para Deakin si está habilitado
    deakin_mac_mapping = {}
    if DATASET_ENABLED.get("Deakin", False):
        csv_path = PATHS.get("Deakin_CSV")
        if csv_path and os.path.exists(csv_path):
            print(f"\n Cargando mapeo MAC para Deakin...")
            deakin_mac_mapping = load_deakin_mac_mapping(csv_path)
    
    for source, path in PATHS.items():
        # Skip CSV paths (no son directorios de PCAPs)
        if source.endswith("_CSV"):
            continue
            
        # Verificar si el dataset está habilitado
        if not DATASET_ENABLED.get(source, True):
            print(f"\n⏭  {source}: DESHABILITADO (cambiar en DATASET_ENABLED para activar)")
            continue
            
        if not os.path.exists(path):
            print(f"\n  {source}: RUTA NO ENCONTRADA -> {path}")
            continue
            
        print(f"\n{'─'*70}")
        print(f" Procesando {source}...")
        print(f" Ruta: {path}")
        
        found_count = 0
        unknown_count = 0
        label_counter = {}
        
        # Determinar si es Deakin (usa mapeo MAC)
        is_deakin = (source == "Deakin")
        
        for root, dirs, files in os.walk(path):
            for f in files:
                if f.endswith(".pcap") or f.endswith(".pcapng"):
                    folder_name = os.path.basename(root)
                    full_path = os.path.join(root, f)
                    
                    if not is_deakin:
                        # Lógica estándar para UNSW e IoT-23
                        label = get_label(f, folder_name)
                        
                        if label != "Unknown":
                            found_count += 1
                            label_counter[label] = label_counter.get(label, 0) + 1
                            
                            if discovery_mode:
                                if found_count <= 5 or found_count % 20 == 0:
                                    print(f"    [{label:12}] {folder_name}/{f[:30]}...")
                            else:
                                try:
                                    pkts = rdpcap(full_path, count=MAX_PKTS)
                                    for p in pkts:
                                        data = sanitize_packet(p)
                                        if data:
                                            X.append(data)
                                            y.append(label)
                                    if verbose and found_count % 5 == 0:
                                        print(f"   Procesados {found_count} archivos...")
                                except Exception as e:
                                    print(f"   Error leyendo {f}: {e}")
                        else:
                            unknown_count += 1
                            if verbose and unknown_count <= 3:
                                print(f"   Sin etiqueta: {folder_name}/{f[:30]}...")
                    
                    else:
                        # Lógica especial para Deakin (mapeo MAC)
                        if not deakin_mac_mapping:
                            print(f"   No hay mapeo MAC para Deakin. Saltando...")
                            break
                        
                        # Limitar archivos procesados en Deakin
                        try:
                            pkts = rdpcap(full_path, count=5000)  # Más pkts por archivo (multi-dispositivo)
                            file_label_counter = {}
                            
                            for p in pkts:
                                label = get_deakin_label(p, deakin_mac_mapping)
                                
                                if label != "Unknown":
                                    file_label_counter[label] = file_label_counter.get(label, 0) + 1
                                    
                                    if not discovery_mode:
                                        data = sanitize_packet(p)
                                        if data:
                                            X.append(data)
                                            y.append(label)
                            
                            if file_label_counter:
                                found_count += 1
                                for lbl, cnt in file_label_counter.items():
                                    label_counter[lbl] = label_counter.get(lbl, 0) + cnt
                                
                                if discovery_mode and found_count <= 10:
                                    print(f"   {f[:40]}: {file_label_counter}")
                                elif verbose and not discovery_mode and found_count % 5 == 0:
                                    print(f"   Procesados {found_count} archivos Deakin...")
                            else:
                                unknown_count += 1
                                
                        except Exception as e:
                            print(f"   Error leyendo {f}: {e}")
        
        # Resumen del dataset
        dataset_stats[source] = {
            "files_found": found_count,
            "unknown": unknown_count,
            "labels": label_counter
        }
        
        if found_count > 0:
            print(f"\n    Resumen {source}:")
            print(f"      Archivos etiquetados: {found_count}")
            print(f"      Sin etiqueta: {unknown_count}")
            print(f"      Distribución: {label_counter}")
        else:
            print(f"    NO SE ENCONTRARON ARCHIVOS ETIQUETABLES")

    # --- RESUMEN GLOBAL ---
    print("\n" + "=" * 70)
    print(" RESUMEN GLOBAL")
    print("=" * 70)
    
    total_files = sum(s["files_found"] for s in dataset_stats.values())
    total_unknown = sum(s["unknown"] for s in dataset_stats.values())
    
    print(f"Total archivos etiquetados: {total_files}")
    print(f"Total sin etiqueta: {total_unknown}")
    total_files = sum(s["files_found"] for s in dataset_stats.values())
    if not discovery_mode:
        X = np.array(X)
        y = np.array(y)
        print(f"\n DATOS CARGADOS:")
        print(f"   Muestras totales: {len(X):,}")
        print(f"   Shape de entrada: {X.shape if len(X)>0 else 'Vacío'}")
        
        if len(y) > 0:
            unique, counts = np.unique(y, return_counts=True)
            print(f"   Clases únicas: {len(unique)}")
            print(f"\n   Distribución por clase:")
            for label, count in sorted(zip(unique, counts), key=lambda x: -x[1]):
                print(f"      {label:15} : {count:6,} muestras ({100*count/len(y):.1f}%)")
        
        print("=" * 70)
        return X, y
    
    return None, None

# --- 6. EJECUCIÓN ---
print("\n Ejecutando modo DESCUBRIMIENTO (verificación de etiquetas)...\n")
load_datasets(discovery_mode=True)

print("\n" + "=" * 70)
print(" VERIFICACIÓN COMPLETA")
print("=" * 70)
print("\n PRÓXIMOS PASOS:")
print("   1. Revisar la distribución de clases arriba")
print("   2. Si estás satisfecho, descomenta la siguiente línea para cargar datos:")
print("   3. X_train, y_train = load_datasets(discovery_mode=False)")
print("\n Datasets habilitados: UNSW, IoT-23, Deakin")
print(" Datasets deshabilitados: ACI (sin mapeo dispositivo→archivo)")
print("   Para habilitarlos, necesitas investigar qué dispositivo es cada archivo.")


## IMPORTANTE: Re-ejecutar esta celda

Si acabas de abrir el notebook o has reiniciado el kernel, **debes ejecutar esta celda primero** para definir todas las variables de configuración necesarias, incluyendo:
- `DEAKIN_MAX_FILES`
- `DEAKIN_MAX_FILES_ALEXA`  
- `DEAKIN_ALEXA_KEYWORD`

Estas variables son necesarias para el nuevo sistema de carga sin data leakage.

## Análisis del Label Mapping - Nueva Taxonomía Granular

### Nueva Estrategia: 17 Categorías Específicas

**Sin entrenamiento cruzado - Enfoque en clasificación específica por dispositivo**

#### Categorías de Dispositivos:

| Categoría | Descripción | Keywords |
|-----------|-------------|----------|
| **Alexa** | Amazon Echo, Echo Dot, Show | echo, alexa, dot, show |
| **GoogleHome** | Google Home, Nest Audio | googlehome, nest_audio |
| **HomePod** | Apple HomePod | homepod |
| **SmartSpeaker** | Otros speakers IoT | speaker, sonos, triby |
| **SecurityCamera** | Cámaras de seguridad | ring, arlo, canary, outdoor |
| **IndoorCamera** | Cámaras de interior | baby, d-link, simcam, yi |
| **MonitorCamera** | Cámaras de monitoreo | pixstar, withings |
| **MotionSensor** | Sensores de movimiento | motion, sensor, aqara |
| **EnvironmentalSensor** | Sensores ambientales | air, weather, smoke, co2 |
| **HealthSensor** | Sensores de salud | blood pressure, scale, sleep |
| **SmartBulb** | Bombillas inteligentes | bulb, light, lifx |
| **SmartPlug** | Enchufes inteligentes | plug, socket, wemo |
| **SmartSwitch** | Interruptores | switch, tuya switch |
| **SmartLock** | Cerraduras inteligentes | lock, august, smartdoor |
| **Hub** | Controladores centrales | hue bridge, hub, gateway |
| **Printer** | Impresoras | printer, hp |
| **Other** | Otros dispositivos | frame, photo, monitor |

### Ventajas de la Nueva Taxonomía

1. **Mayor granularidad**: Alexa se clasifica específicamente vs otros asistentes
2. **Mejor para demos**: Dispositivos específicos como Alexa para demostración
3. **Más preciso**: Cámaras divididas por tipo de uso (seguridad/interior/monitor)
4. **Sensores categorizados**: Por función (movimiento/ambiente/salud)

### Datasets Habilitados

- UNSW: 27 archivos (nombres descriptivos)
- IoT-23: 9 archivos (mapeo manual por carpeta)
- Deakin: ~90 archivos (mapeo MAC-dispositivo)
   
### Estrategia Recomendada

**FASE 1 (ACTUAL)**: Entrenar con UNSW + IoT-23
- Ventaja: Datos limpios y verificables
- ~36 archivos × 2000 paquetes = ~72,000 muestras (con MAX_PKTS=2000)
- Para producción: subir MAX_PKTS a 50,000+ = ~1.8M muestras

**FASE 2 (OPCIONAL)**: Investigar Deakin/ACI
- Revisar papers originales de los datasets
- Buscar archivos de metadata (JSON, logs, CSVs)
- Contactar autores si es necesario

### Próximos Pasos Técnicos

1. Verificar label mapping - COMPLETADO
2. Cargar datos completos
3. Análisis exploratorio (distribución de bytes, longitudes de paquetes)
4. Diseño de arquitectura CNN
5. Entrenamiento y validación cruzada

## Prueba de Carga con 3 Datasets (UNSW + IoT-23 + Deakin)

Ahora que tenemos integrado Deakin con mapeo MAC, hagamos una carga de prueba con pocos archivos de cada dataset.

In [ ]:
# Ejecutar modo discovery actualizado (después de las modificaciones)
print("\n Ejecutando DISCOVERY MODE para ver estadísticas de los 3 datasets...\n")
load_datasets(discovery_mode=True)

## Inspección Manual de Deakin PCAP

Vamos a investigar por qué solo 1 de 119 archivos tiene paquetes con MACs conocidas.

In [ ]:
# Inspeccionar estructura de capas en los PCAPs de Deakin
import glob

# Cargar mapeo MAC primero
csv_path = PATHS.get("Deakin_CSV")
deakin_mac_mapping_local = load_deakin_mac_mapping(csv_path)

deakin_files = sorted(glob.glob("/media/orb/SSD 1TB1/TMA/Deakin_IoT/pcapIoT/*.pcap"))[:5]
print(f" Analizando estructura de paquetes Deakin...\n")

for pcap_file in deakin_files:
    fname = os.path.basename(pcap_file)
    try:
        pkts = rdpcap(pcap_file, count=100)
        print(f" {fname}")
        print(f"   Total paquetes (primeros 100): {len(pkts)}")
        
        if len(pkts) > 0:
            pkt = pkts[0]
            print(f"   Primer paquete:")
            print(f"      Capas: {[layer.name for layer in pkt.layers()]}")
            print(f"      Has Ether: {Ether in pkt}")
            print(f"      Has IP: {IP in pkt}")
            print(f"      Tipo: {pkt.show(dump=True)[:200]}...")
        print()
        
    except Exception as e:
        print(f"    Error: {e}\n")

## TEST FINAL: Cargar Datos con 3 Datasets

In [ ]:
# TEST: Cargar datos completos con UNSW + IoT-23 + Deakin (limitado)
print(" Cargando datos de 3 datasets (esto puede tomar varios minutos)...\n")

X_raw, y_raw = load_datasets(discovery_mode=False, verbose=True)

In [ ]:
# ============================================================
# PASO 2: CARGAR DATOS REALES (Descomenta cuando estés listo)
# ============================================================
#  ADVERTENCIA: Esto puede tomar 5-15 minutos dependiendo de MAX_PKTS
# Con MAX_PKTS=2000 → ~72K muestras
# Con MAX_PKTS=50000 → ~1.8M muestras (recomendado para entrenamiento final)

# X_train, y_train = load_datasets(discovery_mode=False, verbose=True)

## Debugging Tools: Investigar Datasets sin Metadata

Si quieres investigar **Deakin** o **ACI**, usa las funciones de abajo para:
1. Analizar patrones en nombres de archivos
2. Inspeccionar headers de PCAPs
3. Buscar correlaciones con CSVs/documentación

In [ ]:
# ============================================================
# HERRAMIENTAS DE DEBUGGING PARA DATASETS SIN METADATA
# ============================================================

def analyze_pcap_headers(pcap_path, max_pkts=10):
    """
    Analiza headers de un PCAP para inferir tipo de dispositivo.
    Busca patrones en IPs, puertos, protocolos comunes.
    """
    try:
        pkts = rdpcap(pcap_path, count=max_pkts)
        
        print(f"\n{'='*60}")
        print(f" Archivo: {os.path.basename(pcap_path)}")
        print(f"{'='*60}")
        print(f"Total paquetes analizados: {len(pkts)}")
        
        # Estadísticas de protocolos
        protocols = {}
        ports_src = set()
        ports_dst = set()
        ips_src = set()
        ips_dst = set()
        
        for pkt in pkts:
            # Protocolos L3
            if IP in pkt:
                ips_src.add(pkt[IP].src)
                ips_dst.add(pkt[IP].dst)
                
            # Protocolos L4
            if TCP in pkt:
                protocols['TCP'] = protocols.get('TCP', 0) + 1
                ports_src.add(pkt[TCP].sport)
                ports_dst.add(pkt[TCP].dport)
            elif UDP in pkt:
                protocols['UDP'] = protocols.get('UDP', 0) + 1
                ports_src.add(pkt[UDP].sport)
                ports_dst.add(pkt[UDP].dport)
                
        print(f"\nProtocolos: {protocols}")
        print(f"Puertos destino únicos: {sorted(ports_dst)[:10]}...")  # Primeros 10
        print(f"IPs origen únicas: {len(ips_src)}")
        print(f"IPs destino únicas: {len(ips_dst)}")
        
        # Inferencias básicas
        print(f"\n Inferencias:")
        if 443 in ports_dst or 8443 in ports_dst:
            print("   - Tráfico HTTPS detectado (posible cámara/IoT cloud)")
        if 1883 in ports_dst or 8883 in ports_dst:
            print("   - Puerto MQTT detectado (sensores/actuadores IoT)")
        if 80 in ports_dst:
            print("   - HTTP detectado (dispositivo antiguo o local)")
        if 53 in ports_dst:
            print("   - DNS detectado")
        if 5353 in ports_dst:
            print("   - mDNS/Bonjour detectado (descubrimiento local)")
            
    except Exception as e:
        print(f" Error analizando {pcap_path}: {e}")


def scan_dataset_patterns(dataset_path, max_files=20):
    """
    Escanea un dataset y busca patrones en nombres de archivos/carpetas.
    """
    print(f"\n{'='*70}")
    print(f" ESCANEANDO: {dataset_path}")
    print(f"{'='*70}")
    
    file_count = 0
    
    for root, dirs, files in os.walk(dataset_path):
        for f in files:
            if (f.endswith(".pcap") or f.endswith(".pcapng")) and file_count < max_files:
                full_path = os.path.join(root, f)
                folder_name = os.path.basename(root)
                
                print(f"\n[{file_count+1}] {folder_name}/{f}")
                
                # Analizar primer PCAP de cada carpeta
                if file_count < 5:
                    analyze_pcap_headers(full_path, max_pkts=20)
                    
                file_count += 1
                
    print(f"\n{'='*70}")
    print(f"Total archivos encontrados: {file_count}")


# ============================================================
# EJEMPLOS DE USO (Descomenta según necesites)
# ============================================================

# Opción 1: Analizar UN archivo específico de Deakin
# analyze_pcap_headers("/media/orb/SSD 1TB1/TMA/Deakin_IoT/28013234/pcapIoT/52093085_IoT_2023-07-12.pcap", max_pkts=50)

# Opción 2: Escanear TODOS los archivos de Deakin (solo primeros 20)
# scan_dataset_patterns("/media/orb/SSD 1TB1/TMA/Deakin_IoT/28013234/pcapIoT", max_files=20)

# Opción 3: Escanear ACI
# scan_dataset_patterns("/media/orb/SSD 1TB1/TMA/ACI_IoT/pcap_combined/Combined_Pcaps/Benign-Pcaps", max_files=20)

print(" Herramientas de debugging cargadas.")
print(" Descomenta las líneas de arriba para analizar datasets específicos.")

---
# PASO 1: CARGA DE DATOS Y ANÁLISIS EXPLORATORIO (EDA)

En esta sección vamos a:
1. Cargar los datos completos de los datasets habilitados
2. Analizar distribución de clases
3. Explorar características de los datos (longitudes, patrones de bytes)
4. Visualizar estadísticas clave
5. Preparar datos para entrenamiento

In [ ]:
# Instalar paquetes adicionales para análisis y visualización
%pip install seaborn pandas matplotlib scikit-learn -q

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import time

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print(" Bibliotecas de análisis cargadas.")
print(" Configuración de visualización establecida.")

## 1.1 Carga de Datos Completos

**Advertencia**: Esta operación puede tardar **5-15 minutos** dependiendo de:
- `MAX_PKTS`: Número de paquetes por archivo (actual: 2000)
- Velocidad de lectura del disco
- Número de archivos habilitados

**Métricas esperadas**:
- Con `MAX_PKTS=2000`: ~72,000 muestras
- Con `MAX_PKTS=50000`: ~1.8M muestras (para producción)

In [ ]:
# ============================================================
# CARGAR DATOS COMPLETOS
# ============================================================

print(" Iniciando carga de datos...")
print(f" MAX_PKTS por archivo: {MAX_PKTS}")
print(f" MAX_LEN (bytes por paquete): {MAX_LEN}")
print(f" Datasets habilitados: {[k for k,v in DATASET_ENABLED.items() if v]}")
print("\n" + "="*70)

start_time = time.time()

# Cargar datos
X_raw, y_raw = load_datasets(discovery_mode=False, verbose=True)

elapsed_time = time.time() - start_time

print("\n" + "="*70)
print(f" CARGA COMPLETADA en {elapsed_time:.2f} segundos ({elapsed_time/60:.2f} minutos)")
print("="*70)

# Verificar que se cargaron datos
if X_raw is not None and len(X_raw) > 0:
    print(f"\n DATOS CARGADOS:")
    print(f"   Shape de X: {X_raw.shape}")
    print(f"   Shape de y: {y_raw.shape}")
    print(f"   Tipo de X: {X_raw.dtype}")
    print(f"   Tipo de y: {y_raw.dtype}")
    print(f"   Memoria utilizada: {X_raw.nbytes / 1024 / 1024:.2f} MB")
else:
    print(" ERROR: No se cargaron datos. Verifica las rutas y la configuración.")

## 1.2 Análisis de Distribución de Clases

In [ ]:
# ============================================================
# ANÁLISIS DE DISTRIBUCIÓN DE CLASES
# ============================================================

# Contar muestras por clase
class_counts = Counter(y_raw)
class_names = sorted(class_counts.keys())
class_frequencies = [class_counts[c] for c in class_names]
total_samples = len(y_raw)

print(" DISTRIBUCIÓN DE CLASES")
print("="*70)
print(f"{'Clase':<15} {'Muestras':>10} {'Porcentaje':>12} {'Barra'}")
print("-"*70)

for name, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    percentage = 100 * count / total_samples
    bar = "█" * int(percentage / 2)  # Escala visual
    print(f"{name:<15} {count:>10,} {percentage:>11.2f}% {bar}")

print("="*70)
print(f"{'TOTAL':<15} {total_samples:>10,} {100.0:>11.2f}%")

# Calcular desbalanceo
max_class = max(class_counts.values())
min_class = min(class_counts.values())
imbalance_ratio = max_class / min_class

print(f"\n  Ratio de desbalanceo: {imbalance_ratio:.2f}:1 (max/min)")

if imbalance_ratio > 3:
    print("  ADVERTENCIA: Desbalanceo significativo detectado (>3:1)")
    print("   Recomendaciones:")
    print("   - Usar class_weight='balanced' en el modelo")
    print("   - Considerar SMOTE para sobremuestreo")
    print("   - Usar métricas balanceadas (F1-score, balanced accuracy)")
else:
    print(" Distribución relativamente balanceada (<3:1)")

In [ ]:
# ============================================================
# VISUALIZACIÓN: DISTRIBUCIÓN DE CLASES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras
axes[0].bar(class_names, class_frequencies, color=sns.color_palette("husl", len(class_names)), edgecolor='black')
axes[0].set_xlabel('Clase de Dispositivo', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Número de Muestras', fontsize=12, fontweight='bold')
axes[0].set_title('Distribución de Clases - Valores Absolutos', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Añadir etiquetas con valores
for i, (name, count) in enumerate(zip(class_names, class_frequencies)):
    axes[0].text(i, count + max(class_frequencies)*0.02, f'{count:,}', 
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

# Gráfico de pastel
colors = sns.color_palette("husl", len(class_names))
explode = [0.05] * len(class_names)  # Separar ligeramente las secciones

axes[1].pie(class_frequencies, labels=class_names, autopct='%1.1f%%', 
            startangle=90, colors=colors, explode=explode,
            textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Distribución de Clases - Porcentajes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n Gráficos generados: {len(class_names)} clases visualizadas")

## 1.3 Análisis de Características de los Paquetes

In [ ]:
# ============================================================
# ANÁLISIS DE CARACTERÍSTICAS DE PAQUETES
# ============================================================

print(" ANÁLISIS DE CARACTERÍSTICAS DE LOS DATOS")
print("="*70)

# 1. Análisis de bytes no nulos (contenido real vs padding)
non_zero_counts = np.count_nonzero(X_raw, axis=1)
print(f"\n  ANÁLISIS DE PADDING:")
print(f"   Longitud máxima configurada: {MAX_LEN} bytes")
print(f"   Bytes no-nulos (promedio): {non_zero_counts.mean():.2f} bytes")
print(f"   Bytes no-nulos (mediana): {np.median(non_zero_counts):.2f} bytes")
print(f"   Bytes no-nulos (std): {non_zero_counts.std():.2f} bytes")
print(f"   Mínimo: {non_zero_counts.min()} bytes")
print(f"   Máximo: {non_zero_counts.max()} bytes")

padding_percentage = 100 * (1 - non_zero_counts.mean() / MAX_LEN)
print(f"   Padding promedio: {padding_percentage:.2f}%")

# 2. Análisis de valores de bytes
print(f"\n  ANÁLISIS DE VALORES DE BYTES:")
print(f"   Valor mínimo: {X_raw.min()}")
print(f"   Valor máximo: {X_raw.max()}")
print(f"   Promedio: {X_raw.mean():.2f}")
print(f"   Desviación estándar: {X_raw.std():.2f}")

# 3. Bytes más frecuentes (top 10)
print(f"\n  BYTES MÁS FRECUENTES (Top 10):")
byte_counter = Counter(X_raw.flatten())
most_common = byte_counter.most_common(10)
for byte_val, count in most_common:
    percentage = 100 * count / X_raw.size
    print(f"   Byte {byte_val:3d} (0x{byte_val:02x}): {count:>12,} ocurrencias ({percentage:.2f}%)")

# 4. Análisis por clase
print(f"\n  LONGITUDES PROMEDIO POR CLASE:")
print(f"   {'Clase':<15} {'Bytes No-Nulos (promedio)':<30} {'Std':<10}")
print("   " + "-"*60)
for class_name in class_names:
    mask = y_raw == class_name
    class_non_zero = non_zero_counts[mask]
    print(f"   {class_name:<15} {class_non_zero.mean():>20.2f} bytes {class_non_zero.std():>15.2f}")

print("="*70)

In [ ]:
# ============================================================
# VISUALIZACIÓN: DISTRIBUCIÓN DE LONGITUDES DE PAQUETES
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histograma general de longitudes
axes[0, 0].hist(non_zero_counts, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(non_zero_counts.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {non_zero_counts.mean():.0f}')
axes[0, 0].axvline(np.median(non_zero_counts), color='green', linestyle='--', linewidth=2, label=f'Mediana: {np.median(non_zero_counts):.0f}')
axes[0, 0].set_xlabel('Bytes No-Nulos', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Distribución de Longitudes de Paquetes', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Boxplot por clase
class_lengths = [non_zero_counts[y_raw == c] for c in class_names]
bp = axes[0, 1].boxplot(class_lengths, labels=class_names, patch_artist=True)
for patch, color in zip(bp['boxes'], sns.color_palette("husl", len(class_names))):
    patch.set_facecolor(color)
axes[0, 1].set_xlabel('Clase', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Bytes No-Nulos', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Distribución de Longitudes por Clase', fontsize=12, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Heatmap de primeros 100 bytes (muestra aleatoria)
sample_indices = np.random.choice(len(X_raw), min(50, len(X_raw)), replace=False)
sample_data = X_raw[sample_indices, :100]  # Primeros 100 bytes
im = axes[1, 0].imshow(sample_data, cmap='viridis', aspect='auto', interpolation='nearest')
axes[1, 0].set_xlabel('Posición del Byte', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Muestra', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Heatmap de Primeros 100 Bytes (50 muestras)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1, 0], label='Valor del Byte')

# 4. Distribución de valores de bytes
byte_values = X_raw.flatten()
byte_values = byte_values[byte_values > 0]  # Excluir padding
axes[1, 1].hist(byte_values, bins=256, color='coral', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Valor del Byte', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Frecuencia (log)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Distribución de Valores de Bytes (sin padding)', fontsize=12, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(" Visualizaciones generadas correctamente")

## 1.4 Codificación de Etiquetas y División de Datos

In [ ]:
# ============================================================
# CODIFICACIÓN DE ETIQUETAS
# ============================================================

print(" CODIFICACIÓN DE ETIQUETAS")
print("="*70)

# Crear el codificador
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)

# Mostrar mapeo
print("\n Mapeo de Etiquetas:")
for idx, class_name in enumerate(label_encoder.classes_):
    count = np.sum(y_encoded == idx)
    print(f"   {class_name:<15} → {idx} ({count:,} muestras)")

# Guardar mapeo para referencia futura
label_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}
print(f"\n Codificador creado: {len(label_encoder.classes_)} clases")
print("="*70)

In [ ]:
# ============================================================
# DIVISIÓN DE DATOS: TRAIN / VALIDATION / TEST
# ============================================================

print("\n DIVISIÓN DE DATOS")
print("="*70)

# División estratificada para mantener proporciones en cada subset
# 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y_encoded, 
    test_size=0.15, 
    random_state=42, 
    stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=0.1765,  # 0.1765 * 0.85 ≈ 0.15 del total
    random_state=42, 
    stratify=y_temp
)

print(f" División completada:")
print(f"   Training:   {len(X_train):>6,} muestras ({100*len(X_train)/len(X_raw):.1f}%)")
print(f"   Validation: {len(X_val):>6,} muestras ({100*len(X_val)/len(X_raw):.1f}%)")
print(f"   Test:       {len(X_test):>6,} muestras ({100*len(X_test)/len(X_raw):.1f}%)")

# Verificar estratificación
print(f"\n Verificación de Estratificación:")
print(f"   {'Clase':<15} {'Train %':>10} {'Val %':>10} {'Test %':>10}")
print("   " + "-"*50)

for idx, class_name in enumerate(label_encoder.classes_):
    train_pct = 100 * np.sum(y_train == idx) / len(y_train)
    val_pct = 100 * np.sum(y_val == idx) / len(y_val)
    test_pct = 100 * np.sum(y_test == idx) / len(y_test)
    print(f"   {class_name:<15} {train_pct:>9.2f}% {val_pct:>9.2f}% {test_pct:>9.2f}%")

print("="*70)

# Normalizar datos a rango [0, 1]
print(f"\n Normalizando datos a rango [0, 1]...")
X_train_norm = X_train.astype('float32') / 255.0
X_val_norm = X_val.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

print(f" Normalización completada")
print(f"   Rango original: [{X_train.min()}, {X_train.max()}]")
print(f"   Rango normalizado: [{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]")
print("="*70)

## 1.5 Guardar Datos Procesados (Opcional)

In [ ]:
# ============================================================
# GUARDAR DATOS PROCESADOS (para reutilizar sin recargar PCAPs)
# ============================================================

import pickle

SAVE_PROCESSED = True  # Cambiar a False si no quieres guardar

if SAVE_PROCESSED:
    print(" Guardando datos procesados...")
    
    data_dict = {
        'X_train': X_train_norm,
        'X_val': X_val_norm,
        'X_test': X_test_norm,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
        'label_encoder': label_encoder,
        'label_mapping': label_mapping,
        'class_names': class_names,
        'config': {
            'MAX_PKTS': MAX_PKTS,
            'MAX_LEN': MAX_LEN,
            'datasets_used': [k for k, v in DATASET_ENABLED.items() if v]
        }
    }
    
    save_path = os.path.join(DATA_ROOT, 'processed_data.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(data_dict, f, protocol=4)
    
    file_size_mb = os.path.getsize(save_path) / 1024 / 1024
    print(f" Datos guardados en: {save_path}")
    print(f"   Tamaño del archivo: {file_size_mb:.2f} MB")
    print(f"\n Para cargar en el futuro:")
    print(f"   with open('{save_path}', 'rb') as f:")
    print(f"       data = pickle.load(f)")
else:
    print("  Guardado deshabilitado (SAVE_PROCESSED=False)")

## 1.6 Resumen Final del EDA

### Tareas Completadas:
1. Carga de datos desde PCAP files
2. Análisis de distribución de clases
3. Análisis de características de paquetes
4. Visualizaciones estadísticas
5. Codificación de etiquetas
6. División estratificada (70/15/15)
7. Normalización de datos
8. Guardado de datos procesados

### Variables Clave Disponibles:
- `X_train_norm`, `y_train`: Datos de entrenamiento normalizados
- `X_val_norm`, `y_val`: Datos de validación normalizados
- `X_test_norm`, `y_test`: Datos de prueba normalizados
- `label_encoder`: Codificador de etiquetas
- `label_mapping`: Diccionario de mapeo clase-número
- `class_names`: Lista de nombres de clases

### Próximo Paso: Diseño del Modelo CNN
Ahora estás listo para:
1. Definir la arquitectura CNN
2. Compilar el modelo
3. Entrenar con los datos preparados
4. Evaluar rendimiento

In [ ]:
# ============================================================
# RESUMEN ESTADÍSTICO FINAL
# ============================================================

print("\n" + "="*70)
print(" RESUMEN FINAL DEL ANÁLISIS EXPLORATORIO")
print("="*70)

print("\n  DATOS CARGADOS:")
print(f"   • Total muestras: {len(X_raw):,}")
print(f"   • Dimensiones: {X_raw.shape}")
print(f"   • Clases: {len(class_names)}")
print(f"   • Memoria RAM: {X_raw.nbytes / 1024 / 1024:.2f} MB")

print("\n  DIVISIÓN DE DATOS:")
print(f"   • Training:   {len(X_train):>6,} muestras (70%)")
print(f"   • Validation: {len(X_val):>6,} muestras (15%)")
print(f"   • Test:       {len(X_test):>6,} muestras (15%)")

print("\n  BALANCE DE CLASES:")
print(f"   • Ratio de desbalanceo: {imbalance_ratio:.2f}:1")
print(f"   • Clase mayoritaria: Camera ({100*max(class_counts.values())/total_samples:.1f}%)")
print(f"   • Clase minoritaria: Peripheral ({100*min(class_counts.values())/total_samples:.1f}%)")

print("\n  CARACTERÍSTICAS DE PAQUETES:")
print(f"   • Longitud configurada: {MAX_LEN} bytes")
print(f"   • Longitud promedio real: {non_zero_counts.mean():.1f} bytes")
print(f"   • Padding promedio: {padding_percentage:.1f}%")
print(f"   • Rango de valores: [0, 255]")

print("\n  ARCHIVOS GENERADOS:")
print(f"   • processed_data.pkl (135 MB)")
print(f"   • Contiene: X_train, X_val, X_test, y_*, label_encoder")

print("\n" + "="*70)
print(" EDA COMPLETADO - LISTO PARA DISEÑAR EL MODELO")
print("="*70)

print("\n PRÓXIMOS PASOS:")
print("   1. Diseñar arquitectura CNN 1D")
print("   2. Configurar class_weight='balanced' (por desbalanceo)")
print("   3. Definir callbacks (EarlyStopping, ModelCheckpoint)")
print("   4. Entrenar modelo")
print("   5. Evaluar con métricas balanceadas (F1-score)")

print("\n  CONSIDERACIONES IMPORTANTES:")
print("   • Desbalanceo 15:1 → USAR class_weight o SMOTE")
print("   • Mucho padding (74%) → La CNN debe aprender patrones en headers")
print("   • Longitudes variables → MaxPooling ayudará a generalizar")
print("   • 6 clases → Output: Dense(6, activation='softmax')")

---
# PASO 2: DISEÑO Y ENTRENAMIENTO DEL MODELO CNN

En esta sección vamos a:
1. Diseñar la arquitectura CNN 1D optimizada
2. Calcular class weights para manejar el desbalanceo
3. Configurar métricas y callbacks
4. Entrenar el modelo
5. Evaluar rendimiento y analizar resultados

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

print(" TensorFlow y Keras importados")
print(f" TensorFlow version: {tf.__version__}")
print(f" Keras version: {keras.__version__}")

# Verificar GPU
if tf.config.list_physical_devices('GPU'):
    print(f" GPU disponible: {tf.config.list_physical_devices('GPU')}")
else:
    print("  Entrenando en CPU (será más lento)")

## 2.1 Cálculo de Class Weights

Debido al desbalanceo significativo (15:1), debemos usar **class weights** para que el modelo no ignore las clases minoritarias.

In [ ]:
# ============================================================
# CALCULAR CLASS WEIGHTS
# ============================================================

print("  CÁLCULO DE CLASS WEIGHTS")
print("="*70)

# Calcular weights usando sklearn
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convertir a diccionario para Keras
class_weights_dict = {i: weight for i, weight in enumerate(class_weights_array)}

print("\n Class Weights Calculados:")
print(f"   {'Clase':<15} {'ID':>3} {'Muestras':>10} {'Weight':>10}")
print("   " + "-"*50)

for idx, class_name in enumerate(label_encoder.classes_):
    n_samples = np.sum(y_train == idx)
    weight = class_weights_dict[idx]
    print(f"   {class_name:<15} {idx:>3} {n_samples:>10,} {weight:>10.3f}")

print("\n Interpretación:")
print("   • Weight > 1.0: Clase minoritaria (se penaliza más el error)")
print("   • Weight < 1.0: Clase mayoritaria (menos penalización)")
print("   • Weight ≈ 1.0: Clase balanceada")

print("\n Class weights configurados")
print("="*70)

## 2.2 Diseño de la Arquitectura CNN 1D

Arquitectura optimizada para clasificación de paquetes de red:
- **Input**: (500, 1) - Vector de bytes del paquete
- **3 Bloques Conv1D**: Extracción jerárquica de features
- **GlobalMaxPooling**: Captura features más importantes independiente de posición
- **Dense + Dropout**: Clasificación con regularización
- **Output**: 17 clases granulares (Alexa, GoogleHome, SecurityCamera, etc.)

**Nueva taxonomía granular** (sin entrenamiento cruzado):
- Asistentes de voz específicos: Alexa, GoogleHome, HomePod, SmartSpeaker
- Cámaras por tipo: SecurityCamera, IndoorCamera, MonitorCamera
- Sensores por función: MotionSensor, EnvironmentalSensor, HealthSensor
- Control: SmartBulb, SmartPlug, SmartSwitch, SmartLock
- Hub, Printer, Other

In [ ]:
# ============================================================
# CONSTRUIR MODELO CNN 1D
# ============================================================

def build_cnn_model(input_shape=(500, 1), num_classes=17):
    """
    CNN 1D para clasificación granular de tráfico IoT.
    
    Arquitectura:
    - 3 bloques convolucionales con aumento progresivo de filtros
    - Batch normalization para estabilidad
    - GlobalMaxPooling para capturar features clave
    - Dropout para regularización
    
    Nueva taxonomía: 17 clases específicas por dispositivo/función
    """
    
    model = models.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        
        # Bloque 1: Detectar patrones básicos (headers L2-L4)
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same', name='conv1'),
        layers.BatchNormalization(name='bn1'),
        layers.MaxPooling1D(pool_size=2, name='pool1'),
        layers.Dropout(0.2, name='dropout1'),
        
        # Bloque 2: Patrones de nivel medio (combinaciones de bytes)
        layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same', name='conv2'),
        layers.BatchNormalization(name='bn2'),
        layers.MaxPooling1D(pool_size=2, name='pool2'),
        layers.Dropout(0.3, name='dropout2'),
        
        # Bloque 3: Patrones de alto nivel (características complejas)
        layers.Conv1D(filters=256, kernel_size=3, activation='relu', padding='same', name='conv3'),
        layers.BatchNormalization(name='bn3'),
        layers.MaxPooling1D(pool_size=2, name='pool3'),
        layers.Dropout(0.4, name='dropout3'),
        
        # Global pooling: Capturar feature más relevante de toda la secuencia
        layers.GlobalMaxPooling1D(name='global_pool'),
        
        # Capas densas para clasificación
        layers.Dense(128, activation='relu', name='dense1'),
        layers.BatchNormalization(name='bn_dense'),
        layers.Dropout(0.5, name='dropout_dense'),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax', name='output')
    ])
    
    return model

# Crear modelo
print("  CONSTRUYENDO MODELO CNN")
print("="*70)

model = build_cnn_model(input_shape=(MAX_LEN, 1), num_classes=len(class_names))

print(" Modelo creado exitosamente")
print("="*70)

In [ ]:
# Mostrar arquitectura del modelo
print("\n ARQUITECTURA DEL MODELO:")
print("="*70)
model.summary()

# Calcular parámetros
total_params = model.count_params()
print(f"\n Total de parámetros: {total_params:,}")
print(f"   Tamaño estimado del modelo: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")
print("="*70)

## 2.3 Compilación del Modelo y Configuración de Callbacks

In [ ]:
# ============================================================
# COMPILAR MODELO
# ============================================================

print("  COMPILANDO MODELO")
print("="*70)

# Configurar optimizador
optimizer = keras.optimizers.Adam(learning_rate=0.001)

# Compilar modelo
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',  # Para labels sin one-hot encoding
    metrics=['accuracy']
)

print(" Modelo compilado")
print(f"   Optimizer: Adam (lr=0.001)")
print(f"   Loss: sparse_categorical_crossentropy")
print(f"   Metrics: accuracy")
print("="*70)

In [ ]:
# ============================================================
# CONFIGURAR CALLBACKS
# ============================================================

print("\n CONFIGURANDO CALLBACKS")
print("="*70)

# Directorio para guardar modelos
models_dir = os.path.join(DATA_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)

# 1. EarlyStopping: Detener si no hay mejora
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1,
    mode='min'
)
print(" EarlyStopping configurado:")
print("   • Monitor: val_loss")
print("   • Patience: 15 epochs")
print("   • Restore best weights: True")

# 2. ModelCheckpoint: Guardar mejor modelo
checkpoint_path = os.path.join(models_dir, "best_model.keras")
model_checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1,
    mode='max'
)
print(f"\n ModelCheckpoint configurado:")
print(f"   • Path: {checkpoint_path}")
print(f"   • Monitor: val_accuracy")
print(f"   • Save best only: True")

# 3. ReduceLROnPlateau: Reducir learning rate si se estanca
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1,
    mode='min'
)
print(f"\n ReduceLROnPlateau configurado:")
print(f"   • Monitor: val_loss")
print(f"   • Factor: 0.5 (reduce LR a la mitad)")
print(f"   • Patience: 5 epochs")
print(f"   • Min LR: 1e-7")

callbacks_list = [early_stopping, model_checkpoint, reduce_lr]

print("\n Total de callbacks: 3")
print("="*70)

## PROBLEMA DE DATA LEAKAGE IDENTIFICADO

**Problema**: El modelo alcanza 95% accuracy en epoch 1 debido a **data leakage severo**:

### Causa Raíz:
1. **Split incorrecto**: `train_test_split()` divide por **paquetes**, no por **archivos/dispositivos**
2. **Consecuencia**: Paquetes del MISMO dispositivo están en train, validation Y test
3. **Resultado**: El modelo memoriza patrones específicos de cada dispositivo físico (checksums, TTL, timing patterns)

### Por qué NO funciona con Alexa real:
- Training: Vio 10,000 paquetes de `AmazonEcho_44650d56ccd3.pcap`
- Test set: Tiene otros 5,000 paquetes del MISMO archivo
- Alexa real: **Dispositivo completamente nuevo** - Patrones diferentes - NO detectado

### Solución:
**Split por archivo/dispositivo** (file-stratified split):
- Train: 60% de los ARCHIVOS
- Validation: 20% de los ARCHIVOS  
- Test: 20% de los ARCHIVOS

De esta forma, el modelo aprende comportamiento general de la clase, no patrones de dispositivos específicos.

In [ ]:
# ============================================================
# NUEVA VERSIÓN: load_datasets con tracking de archivos
# ============================================================

def load_datasets_with_file_tracking(verbose=True):
    """
    Carga datasets trackando qué archivo generó cada paquete.
    
    Returns:
        tuple: (X, y, file_ids)
            X: array de bytes normalizados
            y: array de labels
            file_ids: array de identificadores de archivo (para split estratificado)
    """
    X, y, file_ids = [], [], []
    file_counter = 0
    
    print("=" * 70)
    print("CARGANDO DATOS CON FILE TRACKING")
    print("=" * 70)
    
    # Cargar mapeo MAC para Deakin si está habilitado
    deakin_mac_mapping = {}
    if DATASET_ENABLED.get("Deakin", False):
        csv_path = PATHS.get("Deakin_CSV")
        if csv_path and os.path.exists(csv_path):
            print(f"\n Cargando mapeo MAC para Deakin...")
            deakin_mac_mapping = load_deakin_mac_mapping(csv_path)
    
    # Limitar archivos Deakin
    deakin_processed = 0
    deakin_alexa_processed = 0
    
    for source, path in PATHS.items():
        if source.endswith("_CSV"):
            continue
        if not DATASET_ENABLED.get(source, True):
            print(f"\n  {source}: DESHABILITADO")
            continue
        if not os.path.exists(path):
            print(f"\n  {source}: NO ENCONTRADO")
            continue
        
        print(f"\n{'─'*70}")
        print(f" {source}")
        
        is_deakin = (source == "Deakin")
        
        for root, dirs, files in os.walk(path):
            for f in files:
                if not (f.endswith(".pcap") or f.endswith(".pcapng")):
                    continue
                
                folder_name = os.path.basename(root)
                full_path = os.path.join(root, f)
                file_id = file_counter
                
                if not is_deakin:
                    # UNSW, IoT-23
                    label = get_label(f, folder_name)
                    if label == "Unknown":
                        continue
                    
                    try:
                        pkts = rdpcap(full_path, count=MAX_PKTS)
                        packets_added = 0
                        for p in pkts:
                            data = sanitize_packet(p)
                            if data:
                                X.append(data)
                                y.append(label)
                                file_ids.append(file_id)
                                packets_added += 1
                        
                        if packets_added > 0:
                            file_counter += 1
                            if file_counter % 10 == 0:
                                print(f"    {file_counter} archivos, {len(X):,} paquetes...")
                    except Exception as e:
                        if verbose:
                            print(f"    Error: {f[:30]}: {e}")
                
                else:
                    # Deakin con límite de archivos
                    is_alexa_file = DEAKIN_ALEXA_KEYWORD.lower() in f.lower()
                    
                    if is_alexa_file:
                        if deakin_alexa_processed >= DEAKIN_MAX_FILES_ALEXA:
                            continue
                    else:
                        if deakin_processed >= DEAKIN_MAX_FILES:
                            continue
                    
                    try:
                        pkts = rdpcap(full_path, count=5000)
                        packets_added = 0
                        
                        for p in pkts:
                            label = get_deakin_label(p, deakin_mac_mapping)
                            if label == "Unknown":
                                continue
                            
                            data = sanitize_packet(p)
                            if data:
                                X.append(data)
                                y.append(label)
                                file_ids.append(file_id)
                                packets_added += 1
                        
                        if packets_added > 0:
                            file_counter += 1
                            if is_alexa_file:
                                deakin_alexa_processed += 1
                            else:
                                deakin_processed += 1
                            
                            if file_counter % 10 == 0:
                                print(f"    {file_counter} archivos, {len(X):,} paquetes...")
                    except Exception as e:
                        if verbose:
                            print(f"    Error: {f[:30]}: {e}")
    
    print("\n" + "=" * 70)
    print(" CARGA COMPLETA")
    print(f"   Total archivos: {file_counter}")
    print(f"   Total paquetes: {len(X):,}")
    print(f"   Archivos únicos: {len(np.unique(file_ids))}")
    print("=" * 70)
    
    return np.array(X), np.array(y), np.array(file_ids)


print(" Función load_datasets_with_file_tracking() creada")
print("   Esta versión trackea de qué archivo viene cada paquete")

In [ ]:
# ============================================================
# SPLIT ESTRATIFICADO POR ARCHIVO (NO por paquete)
# ============================================================

def file_stratified_split(X, y, file_ids, test_size=0.2, val_size=0.2, random_state=42):
    """
    Split datos por ARCHIVO, no por paquete individual.
    
    Esto evita data leakage: paquetes del mismo archivo NO aparecen
    en train y test al mismo tiempo.
    
    Args:
        X: array de features
        y: array de labels
        file_ids: array indicando de qué archivo viene cada muestra
        test_size: proporción para test
        val_size: proporción para validation (del restante tras test)
        random_state: semilla aleatoria
        
    Returns:
        X_train, X_val, X_test, y_train, y_val, y_test
    """
    from sklearn.model_selection import train_test_split
    
    print(" FILE-STRATIFIED SPLIT")
    print("="*70)
    
    # Obtener archivos únicos y sus labels
    unique_files = np.unique(file_ids)
    print(f" Total de archivos únicos: {len(unique_files)}")
    
    # Para cada archivo, determinar su label más frecuente
    file_labels = {}
    for file_id in unique_files:
        mask = (file_ids == file_id)
        labels_in_file = y[mask]
        # Label mayoritario en este archivo
        from collections import Counter
        most_common = Counter(labels_in_file).most_common(1)[0][0]
        file_labels[file_id] = most_common
    
    # Distribuir archivos (no paquetes) en train/val/test
    file_labels_array = np.array([file_labels[fid] for fid in unique_files])
    
    # Verificar si hay clases con muy pocos archivos
    from collections import Counter
    label_counts = Counter(file_labels_array)
    min_samples_per_class = min(label_counts.values())
    
    if min_samples_per_class < 2:
        print(f"  ADVERTENCIA: Algunas clases tienen solo {min_samples_per_class} archivo(s)")
        print(f"   Usando split SIN estratificación para evitar error")
        print(f"   Clases afectadas: {[k for k,v in label_counts.items() if v < 2]}")
        
        # Split sin estratificación
        train_val_files, test_files = train_test_split(
            unique_files,
            test_size=test_size,
            random_state=random_state,
            stratify=None  # Sin estratificación
        )
        
        train_files, val_files = train_test_split(
            train_val_files,
            test_size=val_size / (1 - test_size),
            random_state=random_state,
            stratify=None  # Sin estratificación
        )
    else:
        # Split estratificado normal
        train_val_files, test_files = train_test_split(
            unique_files,
            test_size=test_size,
            random_state=random_state,
            stratify=file_labels_array
        )
        
        train_val_labels = np.array([file_labels[fid] for fid in train_val_files])
        train_files, val_files = train_test_split(
            train_val_files,
            test_size=val_size / (1 - test_size),
            random_state=random_state,
            stratify=train_val_labels
        )
    
    print(f"\n Distribución de archivos:")
    print(f"   Train: {len(train_files)} archivos ({len(train_files)/len(unique_files)*100:.1f}%)")
    print(f"   Val:   {len(val_files)} archivos ({len(val_files)/len(unique_files)*100:.1f}%)")
    print(f"   Test:  {len(test_files)} archivos ({len(test_files)/len(unique_files)*100:.1f}%)")
    
    # Crear máscaras para paquetes
    train_mask = np.isin(file_ids, train_files)
    val_mask = np.isin(file_ids, val_files)
    test_mask = np.isin(file_ids, test_files)
    
    # Separar datos
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    
    print(f"\n Distribución de paquetes:")
    print(f"   Train: {len(X_train):,} paquetes ({len(X_train)/len(X)*100:.1f}%)")
    print(f"   Val:   {len(X_val):,} paquetes ({len(X_val)/len(X)*100:.1f}%)")
    print(f"   Test:  {len(X_test):,} paquetes ({len(X_test)/len(X)*100:.1f}%)")
    
    # Verificar que NO hay overlap de archivos
    assert len(set(train_files) & set(val_files)) == 0, " Overlap train-val!"
    assert len(set(train_files) & set(test_files)) == 0, " Overlap train-test!"
    assert len(set(val_files) & set(test_files)) == 0, " Overlap val-test!"
    print(f"\n VERIFICACIÓN: Sin overlap de archivos entre splits")
    
    print("="*70)
    
    return X_train, X_val, X_test, y_train, y_val, y_test


print(" Función file_stratified_split() creada")
print("   Garantiza que archivos completos están en train, val O test (no mezclados)")

## CÓMO USAR EL NUEVO SISTEMA (Sin Data Leakage)

### Método ANTIGUO (con leakage):
```python
# PROBLEMA: Mezcla paquetes del mismo archivo en train/test
X_raw, y_raw = load_datasets(discovery_mode=False)
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, stratify=y_raw)
```

**Resultado**: 95% accuracy epoch 1, pero NO detecta dispositivos nuevos

---

### Método NUEVO (correcto):
```python
# PASO 1: Cargar con tracking de archivos
X_raw, y_raw, file_ids = load_datasets_with_file_tracking()

# PASO 2: Split estratificado POR ARCHIVO
X_train, X_val, X_test, y_train, y_val, y_test = file_stratified_split(
    X_raw, y_raw, file_ids,
    test_size=0.2,
    val_size=0.2,
    random_state=42
)

# PASO 3: Encode labels
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# PASO 4: Normalizar y reshape
X_train_norm = X_train.astype('float32') / 255.0
X_val_norm = X_val.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

X_train_cnn = X_train_norm.reshape(-1, MAX_LEN, 1)
X_val_cnn = X_val_norm.reshape(-1, MAX_LEN, 1)
X_test_cnn = X_test_norm.reshape(-1, MAX_LEN, 1)
```

**Resultado esperado**: 
- Accuracy inicial menor (50-70% epoch 1)
- Mejor generalización a dispositivos nuevos
- Modelo aprende comportamiento de clase, no de dispositivo específico

---

### PRÓXIMOS PASOS:
1. **Ejecutar la celda siguiente** para cargar datos con el nuevo método
2. **Entrenar modelo** - verás accuracy inicial más baja (esperado)
3. **Probar con Alexa real** - ahora debería funcionar mejor

In [ ]:
# ============================================================
# PIPELINE COMPLETO: CARGA Y SPLIT SIN DATA LEAKAGE
# ============================================================

print(" INICIANDO CARGA DE DATOS (Método Correcto - Sin Leakage)")
print("="*70)

# PASO 1: Cargar datos con file tracking
print("\n PASO 1: Cargando datos con tracking de archivos...")
X_raw, y_raw, file_ids = load_datasets_with_file_tracking(verbose=True)

print(f"\n Datos cargados:")
print(f"   Shape X: {X_raw.shape}")
print(f"   Shape y: {y_raw.shape}")
print(f"   Archivos únicos: {len(np.unique(file_ids))}")

# PASO 2: Split estratificado por archivo
print("\n PASO 2: Split estratificado por ARCHIVO (no por paquete)...")
X_train, X_val, X_test, y_train, y_val, y_test = file_stratified_split(
    X_raw, y_raw, file_ids,
    test_size=0.2,
    val_size=0.2,
    random_state=42
)

# PASO 3: Encode labels (FIT en todos los datos para evitar unseen labels)
print("\n  PASO 3: Encoding labels...")
label_encoder = LabelEncoder()
# Fit en TODAS las clases únicas (train + val + test)
all_classes = np.unique(np.concatenate([y_train, y_val, y_test]))
label_encoder.fit(all_classes)

# Ahora transformar cada set
y_train = label_encoder.transform(y_train)
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

print(f"   Clases totales: {list(label_encoder.classes_)}")
print(f"   Número de clases: {len(label_encoder.classes_)}")
print(f"   Clases en train: {len(np.unique(y_train))}")
print(f"   Clases en val: {len(np.unique(y_val))}")
print(f"   Clases en test: {len(np.unique(y_test))}")

# PASO 4: Normalizar y reshape
print("\n PASO 4: Normalizando y reshape para CNN...")
X_train_norm = X_train.astype('float32') / 255.0
X_val_norm = X_val.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

X_train_cnn = X_train_norm.reshape(-1, MAX_LEN, 1)
X_val_cnn = X_val_norm.reshape(-1, MAX_LEN, 1)
X_test_cnn = X_test_norm.reshape(-1, MAX_LEN, 1)

print(f"   X_train_cnn: {X_train_cnn.shape}")
print(f"   X_val_cnn: {X_val_cnn.shape}")
print(f"   X_test_cnn: {X_test_cnn.shape}")

# PASO 5: Calcular class weights
print("\n  PASO 5: Calculando class weights...")
# Solo calcular weights para clases que EXISTEN en train
train_classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=train_classes,
    y=y_train
)
# Mapear a diccionario con TODOS los índices de clases
class_weights_dict = {}
for idx in range(len(label_encoder.classes_)):
    if idx in train_classes:
        # Clase presente en train, usar su weight
        train_idx = np.where(train_classes == idx)[0][0]
        class_weights_dict[idx] = class_weights_array[train_idx]
    else:
        # Clase NO en train (solo en val/test), usar weight neutro
        class_weights_dict[idx] = 1.0

print(f"   Class weights calculados para {len(class_weights_dict)} clases")
print(f"   Clases con weight balanceado: {len(train_classes)}")
print(f"   Clases con weight neutro (no en train): {len(label_encoder.classes_) - len(train_classes)}")

# RESUMEN FINAL
print("\n" + "="*70)
print(" PREPARACIÓN COMPLETA")
print("="*70)
print(f" Distribución final:")
print(f"   Train: {len(X_train_cnn):,} muestras")
print(f"   Val:   {len(X_val_cnn):,} muestras")
print(f"   Test:  {len(X_test_cnn):,} muestras")
print(f"\n LISTO PARA ENTRENAR")
print("   • Sin data leakage")
print("   • Split por archivo/dispositivo")
print("   • Mejor generalización esperada")
print("="*70)

In [ ]:
# ============================================================
# CONSTRUIR MODELO (después de conocer el número de clases)
# ============================================================

print("\n  CONSTRUYENDO MODELO CNN SIMPLIFICADO")
print("="*70)

# Ahora conocemos el número exacto de clases
num_classes = len(label_encoder.classes_)
print(f"   Número de clases: {num_classes}")

# MODELO ULTRA-SIMPLIFICADO - Solo 1 capa Conv (casi lineal)
def build_cnn_model(input_shape=(500, 1), num_classes=14):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # UNA SOLA capa convolucional
        layers.Conv1D(filters=64, kernel_size=7, activation='relu', padding='same', name='conv1'),
        layers.MaxPooling1D(pool_size=4, name='pool1'),
        layers.Dropout(0.3, name='dropout1'),
        
        # Clasificador muy simple
        layers.Flatten(name='flatten'),
        layers.Dense(64, activation='relu', name='dense1'),
        layers.Dropout(0.5, name='dropout_dense'),
        
        # Output
        layers.Dense(num_classes, activation='softmax', name='output')
    ])
    return model

# Crear modelo
model = build_cnn_model(input_shape=(MAX_LEN, 1), num_classes=num_classes)

# Compilar con lr normal
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("\n Modelo ULTRA-SIMPLIFICADO compilado")
print("   - 1 SOLA capa Conv (64 filtros) - Mínima complejidad")
print("   - MaxPooling aggressive (pool_size=4)")
print("   - Solo 64 neuronas en Dense (antes 128)")
print("   - ~90% menos parámetros que modelo original")
print("="*70)

In [ ]:
# ============================================================
# CONFIGURAR CALLBACKS
# ============================================================

print("\n CONFIGURANDO CALLBACKS")
print("="*70)

# Directorio para guardar modelos
models_dir = os.path.join(DATA_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)

# 1. EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1,
    mode='min'
)

# 2. ModelCheckpoint
checkpoint_path = os.path.join(models_dir, "best_model_fixed.keras")
model_checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1,
    mode='max'
)

# 3. ReduceLROnPlateau
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1,
    mode='min'
)

callbacks_list = [early_stopping, model_checkpoint, reduce_lr]

print(" Callbacks configurados:")
print(f"   • EarlyStopping (patience=15)")
print(f"   • ModelCheckpoint: {checkpoint_path}")
print(f"   • ReduceLROnPlateau (factor=0.5, patience=5)")
print("="*70)

In [ ]:
# ============================================================
# ENTRENAR MODELO (Sin Data Leakage)
# ============================================================

print("\n INICIANDO ENTRENAMIENTO")
print("="*70)

print(f" Datos de entrada:")
print(f"   X_train: {X_train_cnn.shape}")
print(f"   X_val: {X_val_cnn.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_val: {y_val.shape}")

# Hiperparámetros
EPOCHS = 50
BATCH_SIZE = 64

print(f"\n  Configuración:")
print(f"   • Epochs: {EPOCHS}")
print(f"   • Batch size: {BATCH_SIZE}")
print(f"   • Class weights:  Activado")
print(f"   • Early stopping:  patience=15")

print("\n EXPECTATIVA CON FIX:")
print("   • Epoch 1: ~50-70% accuracy (normal, no memorización)")
print("   • Entrenamiento: Mejora gradual hasta ~80-90%")
print("   • Generalización: Debería funcionar con dispositivos nuevos")

print("\n" + "="*70)
print(" ENTRENAMIENTO EN PROGRESO...")
print("="*70 + "\n")

# Entrenar
start_time = time.time()

history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights_dict,
    callbacks=callbacks_list,
    verbose=1
)

training_time = time.time() - start_time

print("\n" + "="*70)
print(f" ENTRENAMIENTO COMPLETADO")
print("="*70)
print(f"  Tiempo: {training_time:.2f}s ({training_time/60:.2f}min)")
print(f" Epochs: {len(history.history['loss'])}")
print(f" Mejor val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(f" Mejor val_loss: {min(history.history['val_loss']):.4f}")
print("="*70)

In [ ]:
# ============================================================
# ANÁLISIS DETALLADO DE LA PROGRESIÓN
# ============================================================

print("\n ANÁLISIS COMPLETO DE ENTRENAMIENTO")
print("="*70)

# Obtener número de epochs
n_epochs = len(history.history['accuracy'])
print(f" Epochs completados: {n_epochs}")

# Mostrar progresión detallada
print("\n EVOLUCIÓN EPOCH POR EPOCH:")
print("-"*70)
print(f"{'Epoch':<8} {'Train Acc':>12} {'Val Acc':>12} {'Gap':>10} {'Estado'}")
print("-"*70)

# Mostrar primeros 10 epochs
max_show = min(10, n_epochs)
for i in range(max_show):
    train_acc = history.history['accuracy'][i]
    val_acc = history.history['val_accuracy'][i]
    gap = train_acc - val_acc
    
    # Estado del gap
    if abs(gap) < 0.05:
        estado = " Balanceado"
    elif abs(gap) < 0.10:
        estado = "  Gap moderado"
    else:
        estado = " Gap grande"
    
    print(f"Epoch {i+1:<3} {train_acc:>12.4f} {val_acc:>12.4f} {gap:>+10.4f} {estado}")

if n_epochs > 10:
    print(f"... ({n_epochs - 10} epochs más)")

print("-"*70)

# Análisis de overfitting
final_train = history.history['accuracy'][-1]
final_val = history.history['val_accuracy'][-1]
final_gap = final_train - final_val

print("\n DIAGNÓSTICO DE DATA LEAKAGE:")
print("-"*70)
print(f"Train accuracy final: {final_train:.4f}")
print(f"Val accuracy final:   {final_val:.4f}")
print(f"Gap:                  {final_gap:+.4f} ({abs(final_gap)*100:.2f}%)")

# Verificar primer epoch (indicador clave)
first_train = history.history['accuracy'][0]
first_val = history.history['val_accuracy'][0]

print(f"\n EPOCH 1 (Indicador de Data Leakage):")
print(f"   Train: {first_train:.4f}")
print(f"   Val:   {first_val:.4f}")

if first_train > 0.90 or first_val > 0.90:
    print("     SOSPECHOSO: >90% en epoch 1 sugiere memorización")
elif first_train > 0.70 or first_val > 0.70:
    print("     ALTO: 70-90% en epoch 1 puede indicar algo de leakage")
else:
    print("    NORMAL: <70% en epoch 1 indica aprendizaje genuino")

# Comparación con entrenamiento ANTERIOR (con leakage)
print("\n COMPARACIÓN CON MODELO ANTERIOR (con data leakage):")
print("-"*70)
print("Modelo ANTERIOR (CON leakage):")
print("   Epoch 1:  Train=72%, Val=85%  ← Sospechosamente alto")
print("   Epoch 35: Train=97%, Val=98%  ← Casi perfecto = memorización")
print()
print("Modelo ACTUAL (SIN leakage - esperado):")
print(f"   Epoch 1:  Train={first_train:.0%}, Val={first_val:.0%}")
print(f"   Epoch {n_epochs}:  Train={final_train:.0%}, Val={final_val:.0%}")

# Veredicto
print("\n" + "="*70)
print(" VEREDICTO FINAL:")
print("="*70)

if first_train < 0.70 and abs(final_gap) < 0.10:
    print(" ENTRENAMIENTO SALUDABLE")
    print("   • Epoch 1 bajo: NO hay memorización inmediata")
    print("   • Gap final pequeño: Buena generalización")
    print("   • Conclusión: Fix de data leakage FUNCIONÓ correctamente")
elif first_train > 0.85:
    print(" POSIBLE DATA LEAKAGE PERSISTENTE")
    print("   • Accuracy demasiado alta en epoch 1")
    print("   • Revisa el split estratificado")
else:
    print("  ENTRENAMIENTO ACEPTABLE")
    print("   • Mejora progresiva observable")
    print("   • Puede necesitar más epochs o ajustes")

print("="*70)

## 2.4 Entrenamiento del Modelo

**Configuración de entrenamiento:**
- **Epochs**: 100 (con early stopping)
- **Batch size**: 64
- **Class weights**: Activado (para balancear clases)
- **Validation**: 15% de los datos

**Nota**: Este proceso puede tomar **15-30 minutos** dependiendo del hardware.

In [ ]:
# ============================================================
# ENTRENAR MODELO
# ============================================================

print("\n INICIANDO ENTRENAMIENTO")
print("="*70)

# Preparar datos (añadir dimensión de canal)
X_train_cnn = X_train_norm.reshape(-1, MAX_LEN, 1)
X_val_cnn = X_val_norm.reshape(-1, MAX_LEN, 1)

print(f" Shapes de entrada:")
print(f"   X_train: {X_train_cnn.shape}")
print(f"   X_val: {X_val_cnn.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_val: {y_val.shape}")

# Hiperparámetros
EPOCHS = 100
BATCH_SIZE = 64

print(f"\n  Configuración:")
print(f"   • Epochs: {EPOCHS}")
print(f"   • Batch size: {BATCH_SIZE}")
print(f"   • Batches por epoch (train): {len(X_train_cnn) // BATCH_SIZE}")
print(f"   • Batches por epoch (val): {len(X_val_cnn) // BATCH_SIZE}")
print(f"   • Class weights:  Activado")

print("\n" + "="*70)
print(" ENTRENAMIENTO EN PROGRESO...")
print("="*70 + "\n")

# Entrenar modelo
start_time = time.time()

history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights_dict,
    callbacks=callbacks_list,
    verbose=1
)

training_time = time.time() - start_time

print("\n" + "="*70)
print(f" ENTRENAMIENTO COMPLETADO")
print("="*70)
print(f"  Tiempo total: {training_time:.2f} segundos ({training_time/60:.2f} minutos)")
print(f" Epochs ejecutados: {len(history.history['loss'])}")
print(f" Mejor val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(f" Mejor val_loss: {min(history.history['val_loss']):.4f}")
print("="*70)

## 2.5 Visualización de Curvas de Entrenamiento

In [ ]:
# ============================================================
# VISUALIZAR CURVAS DE ENTRENAMIENTO
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, marker='o', markersize=3)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, marker='s', markersize=3)
axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=11)
axes[0].grid(alpha=0.3)

# Marcar mejor epoch
best_epoch_acc = np.argmax(history.history['val_accuracy'])
best_acc = history.history['val_accuracy'][best_epoch_acc]
axes[0].axvline(x=best_epoch_acc, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch: {best_epoch_acc}')
axes[0].plot(best_epoch_acc, best_acc, 'r*', markersize=15, label=f'Best Val Acc: {best_acc:.4f}')
axes[0].legend(loc='lower right', fontsize=10)

# Gráfico 2: Loss
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2, marker='o', markersize=3)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2, marker='s', markersize=3)
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Loss', fontsize=12, fontweight='bold')
axes[1].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[1].legend(loc='upper right', fontsize=11)
axes[1].grid(alpha=0.3)

# Marcar mejor epoch
best_epoch_loss = np.argmin(history.history['val_loss'])
best_loss = history.history['val_loss'][best_epoch_loss]
axes[1].axvline(x=best_epoch_loss, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch: {best_epoch_loss}')
axes[1].plot(best_epoch_loss, best_loss, 'r*', markersize=15, label=f'Best Val Loss: {best_loss:.4f}')
axes[1].legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

# Análisis de overfitting/underfitting
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
gap = final_train_acc - final_val_acc

print("\n ANÁLISIS DE CURVAS:")
print("="*70)
print(f"Accuracy final (train): {final_train_acc:.4f}")
print(f"Accuracy final (val):   {final_val_acc:.4f}")
print(f"Gap (train - val):      {gap:.4f}")

if gap > 0.1:
    print("\n  OVERFITTING DETECTADO (gap > 0.1)")
    print("   Recomendaciones:")
    print("   • Aumentar dropout")
    print("   • Reducir complejidad del modelo")
    print("   • Usar data augmentation")
elif gap < 0.02:
    print("\n MODELO BIEN BALANCEADO (gap < 0.02)")
    print("   El modelo generaliza correctamente")
else:
    print("\n OVERFITTING LEVE (0.02 < gap < 0.1)")
    print("   Aceptable para esta fase del proyecto")

print("="*70)

## 2.6 Evaluación en Test Set

In [ ]:
# ============================================================
# EVALUACIÓN EN TEST SET
# ============================================================

print("\n EVALUACIÓN EN TEST SET")
print("="*70)

# Preparar test data
X_test_cnn = X_test_norm.reshape(-1, MAX_LEN, 1)

# Evaluar modelo
test_loss, test_accuracy = model.evaluate(X_test_cnn, y_test, verbose=0)

print(f"\n Métricas en Test Set:")
print(f"   Test Loss:     {test_loss:.4f}")
print(f"   Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Predicciones
y_pred = model.predict(X_test_cnn, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

# Calcular F1-scores
f1_macro = f1_score(y_test, y_pred_classes, average='macro')
f1_weighted = f1_score(y_test, y_pred_classes, average='weighted')

print(f"\n F1-Scores:")
print(f"   F1-Score (macro):    {f1_macro:.4f}")
print(f"   F1-Score (weighted): {f1_weighted:.4f}")

print("\n Evaluación completada")
print("="*70)

## 2.7 Classification Report y Métricas por Clase

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n CLASSIFICATION REPORT (Test Set)")
print("="*70)

# Generar reporte
report = classification_report(
    y_test, 
    y_pred_classes, 
    target_names=label_encoder.classes_,
    digits=4
)
print(report)

# Métricas por clase en formato tabla
print("\n MÉTRICAS DETALLADAS POR CLASE:")
print("="*70)

from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred_classes, average=None, labels=range(len(class_names))
)

print(f"{'Clase':<15} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-"*70)

for idx, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name:<15} {precision[idx]:>10.4f} {recall[idx]:>10.4f} {f1[idx]:>10.4f} {int(support[idx]):>10,}")

print("="*70)

# Análisis de clases problemáticas
print("\n ANÁLISIS DE RENDIMIENTO:")
print("-"*70)

# Clase con mejor F1-score
best_class_idx = np.argmax(f1)
best_class = label_encoder.classes_[best_class_idx]
print(f" Mejor clase: {best_class} (F1={f1[best_class_idx]:.4f})")

# Clase con peor F1-score
worst_class_idx = np.argmin(f1)
worst_class = label_encoder.classes_[worst_class_idx]
print(f"  Peor clase: {worst_class} (F1={f1[worst_class_idx]:.4f})")

# Clases con buen balance precision/recall
balanced_classes = [label_encoder.classes_[i] for i in range(len(class_names)) 
                    if abs(precision[i] - recall[i]) < 0.05]
print(f"\n  Clases balanceadas (|P-R| < 0.05): {', '.join(balanced_classes)}")

# Clases con alta precision pero bajo recall (falsos negativos)
fn_problem = [label_encoder.classes_[i] for i in range(len(class_names)) 
              if precision[i] > recall[i] + 0.1]
if fn_problem:
    print(f" Clases con problemas de detección (FN): {', '.join(fn_problem)}")
    print(f"   → El modelo NO detecta todos los casos")

# Clases con alto recall pero baja precision (falsos positivos)
fp_problem = [label_encoder.classes_[i] for i in range(len(class_names)) 
              if recall[i] > precision[i] + 0.1]
if fp_problem:
    print(f" Clases con problemas de especificidad (FP): {', '.join(fp_problem)}")
    print(f"   → El modelo predice esta clase incorrectamente")

print("="*70)

## 2.8 Confusion Matrix (Matriz de Confusión)

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

# Calcular matriz de confusión
cm = confusion_matrix(y_test, y_pred_classes)

# Crear figura con 2 subgráficos
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Matriz de confusión (valores absolutos)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Número de muestras'},
            ax=axes[0])
axes[0].set_xlabel('Predicción', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Etiqueta Real', fontsize=12, fontweight='bold')
axes[0].set_title('Confusion Matrix (Valores Absolutos)', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Matriz de confusión normalizada (porcentajes)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Porcentaje'},
            ax=axes[1])
axes[1].set_xlabel('Predicción', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Etiqueta Real', fontsize=12, fontweight='bold')
axes[1].set_title('Confusion Matrix Normalizada (% por fila)', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

# Análisis de confusiones
print("\n ANÁLISIS DE CONFUSIONES MÁS FRECUENTES:")
print("="*70)

# Encontrar las confusiones más grandes (excluyendo diagonal)
confusions = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confusions.append((
                label_encoder.classes_[i],
                label_encoder.classes_[j],
                cm[i, j],
                cm_normalized[i, j]
            ))

# Ordenar por número de confusiones
confusions_sorted = sorted(confusions, key=lambda x: x[2], reverse=True)

# Mostrar top 10 confusiones
print(f"{'Real':<15} {'→ Predicho':<15} {'Casos':>8} {'% de la clase':>15}")
print("-"*70)
for real, pred, count, pct in confusions_sorted[:10]:
    print(f"{real:<15} → {pred:<15} {int(count):>8} {pct:>14.2%}")

print("="*70)

# Calcular accuracy en diagonal
diagonal_sum = np.trace(cm)
total_sum = np.sum(cm)
print(f"\n Predicciones correctas (diagonal): {diagonal_sum:,} / {total_sum:,} ({diagonal_sum/total_sum:.2%})")
print(f" Predicciones incorrectas (fuera diagonal): {total_sum - diagonal_sum:,} ({(total_sum - diagonal_sum)/total_sum:.2%})")
print("="*70)

## 2.9 Guardar Resultados y Resumen Final

In [ ]:
# ============================================================
# GUARDAR RESULTADOS Y MODELO
# ============================================================

print("\n GUARDANDO RESULTADOS")
print("="*70)

# Guardar historial de entrenamiento
history_path = os.path.join(models_dir, 'training_history.pkl')
with open(history_path, 'wb') as f:
    pickle.dump(history.history, f)
print(f" Historial guardado: {history_path}")

# Guardar métricas de test
results = {
    'test_accuracy': test_accuracy,
    'test_loss': test_loss,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted,
    'confusion_matrix': cm,
    'classification_report': report,
    'class_names': label_encoder.classes_.tolist(),
    'training_time': training_time,
    'epochs_trained': len(history.history['loss']),
    'best_val_accuracy': max(history.history['val_accuracy']),
    'best_val_loss': min(history.history['val_loss'])
}

results_path = os.path.join(models_dir, 'test_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(results, f)
print(f" Resultados guardados: {results_path}")

print(f" Mejor modelo guardado: {checkpoint_path}")
print("="*70)

# RESUMEN FINAL
print("\n" + "="*70)
print(" RESUMEN FINAL DEL ENTRENAMIENTO")
print("="*70)

print(f"\n DATOS:")
print(f"   • Total muestras: {len(X_raw):,}")
print(f"   • Training: {len(X_train):,} ({len(X_train)/len(X_raw)*100:.1f}%)")
print(f"   • Validation: {len(X_val):,} ({len(X_val)/len(X_raw)*100:.1f}%)")
print(f"   • Test: {len(X_test):,} ({len(X_test)/len(X_raw)*100:.1f}%)")

print(f"\n  MODELO:")
print(f"   • Arquitectura: CNN 1D (3 bloques)")
print(f"   • Parámetros: {total_params:,}")
print(f"   • Input shape: ({MAX_LEN}, 1)")
print(f"   • Output classes: {len(class_names)}")

print(f"\n  ENTRENAMIENTO:")
print(f"   • Tiempo total: {training_time/60:.2f} minutos")
print(f"   • Epochs ejecutados: {len(history.history['loss'])}")
print(f"   • Batch size: {BATCH_SIZE}")
print(f"   • Class weights:  Activado")

print(f"\n RENDIMIENTO EN TEST SET:")
print(f"   • Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   • F1-Score (macro): {f1_macro:.4f}")
print(f"   • F1-Score (weighted): {f1_weighted:.4f}")

print(f"\n ARCHIVOS GENERADOS:")
print(f"   • {checkpoint_path}")
print(f"   • {history_path}")
print(f"   • {results_path}")

print("\n" + "="*70)
print(" PASO 2 COMPLETADO - MODELO ENTRENADO Y EVALUADO")
print("="*70)

print("\n PRÓXIMOS PASOS SUGERIDOS:")
print("   1. Analizar confusiones más frecuentes")
print("   2. Probar con más datos (aumentar MAX_PKTS)")
print("   3. Experimentar con arquitecturas alternativas")
print("   4. Implementar data augmentation")
print("   5. Optimizar para inferencia rápida (<50ms)")
print("   6. Integrar con firewall experimental")

In [ ]:
# Añade esta celda DESPUÉS de entrenar el modelo (después de la celda 55)

# ============================================================
# EXPORTAR ARCHIVOS PARA INFERENCIA EN PRODUCCIÓN
# ============================================================

import pickle
import json

print(" EXPORTANDO ARCHIVOS PARA INFERENCIA")
print("="*70)

# Directorio de exportación
inference_dir = os.path.join(DATA_ROOT, "inference")
os.makedirs(inference_dir, exist_ok=True)

# 1. Copiar modelo entrenado
import shutil
model_src = os.path.join(models_dir, "best_model.keras")
model_dst = os.path.join(inference_dir, "best_model.keras")
shutil.copy(model_src, model_dst)
print(f" Modelo copiado: {model_dst}")

# 2. Guardar label encoder
encoder_path = os.path.join(inference_dir, "label_encoder.pkl")
with open(encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f" Label encoder guardado: {encoder_path}")

# 3. Guardar configuración del modelo
config = {
    'MAX_LEN': MAX_LEN,
    'MAX_PKTS': MAX_PKTS,
    'class_names': label_encoder.classes_.tolist(),
    'num_classes': len(label_encoder.classes_),
    'model_architecture': {
        'input_shape': (MAX_LEN, 1),
        'total_params': model.count_params()
    },
    'training_info': {
        'test_accuracy': float(test_accuracy),
        'f1_macro': float(f1_macro),
        'training_date': time.strftime('%Y-%m-%d %H:%M:%S')
    }
}

config_path = os.path.join(inference_dir, "model_config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f" Configuración guardada: {config_path}")

# 4. Crear archivo README con instrucciones
readme_content = f"""
#  IoT Device Classifier - Inference Package

##  Archivos Incluidos:
- `best_model.keras`: Modelo CNN entrenado
- `label_encoder.pkl`: Codificador de etiquetas
- `model_config.json`: Configuración del modelo
- `classify_pcap.py`: Script de inferencia

##  Clases que puede detectar:
{chr(10).join([f'  - {name}' for name in label_encoder.classes_])}

##  Rendimiento:
- Accuracy: {test_accuracy*100:.2f}%
- F1-Score: {f1_macro*100:.2f}%

##  Uso:
```bash
python classify_pcap.py <archivo.pcap>
```

##  Requisitos:
```bash
pip install tensorflow scapy numpy
```
"""

readme_path = os.path.join(inference_dir, "README.md")
with open(readme_path, 'w') as f:
    f.write(readme_content)
print(f" README creado: {readme_path}")

print("\n" + "="*70)
print(f" EXPORTACIÓN COMPLETADA")
print(f" Directorio: {inference_dir}")
print("\n Archivos exportados:")
print(f"   1. best_model.keras")
print(f"   2. label_encoder.pkl")
print(f"   3. model_config.json")
print(f"   4. README.md")
print("="*70)

# PASO 3: VALIDACIÓN CRUZADA ENTRE DATASETS

En esta sección probaremos la **generalización del modelo** entrenando con algunos datasets y evaluando en otros no vistos.

**Objetivo:** Verificar que el modelo puede clasificar correctamente dispositivos de datasets completamente diferentes a los usados en entrenamiento.

**Estrategias de validación:**
1. **Train: UNSW+Deakin - Test: IoT-23** (probar generalización a honeypot data)
2. **Train: UNSW+IoT-23 - Test: Deakin** (probar con tráfico real mezclado)
3. **Train: Deakin+IoT-23 - Test: UNSW** (probar con dataset más limpio)

## 3.1 Configuración de Datasets para Validación Cruzada

Configura qué datasets usar para entrenamiento y cuáles para testing.

In [ ]:
# ===============================================================
# CONFIGURACIÓN DE VALIDACIÓN CRUZADA
# ===============================================================

# Selecciona la estrategia de validación cruzada descomentando UNA de las opciones:

# OPCIÓN 1: Entrenar con UNSW+Deakin, Testear con IoT-23
# (Evaluar generalización a datos de honeypot)
# CROSS_VAL_CONFIG = {
#     "strategy": "UNSW+Deakin → IoT-23",
#     "train_datasets": ["UNSW", "Deakin"],
#     "test_datasets": ["IoT23"]
# }

# OPCIÓN 2: Entrenar con UNSW+IoT-23, Testear con Deakin
# (Evaluar generalización a tráfico real multi-dispositivo)
# CROSS_VAL_CONFIG = {
#     "strategy": "UNSW+IoT-23 → Deakin",
#     "train_datasets": ["UNSW", "IoT23"],
#     "test_datasets": ["Deakin"]
# }

# OPCIÓN 3: Entrenar con Deakin+IoT-23, Testear con UNSW
# (Evaluar generalización al dataset más limpio)
CROSS_VAL_CONFIG = {
    "strategy": "Deakin+IoT-23 → UNSW",
    "train_datasets": ["Deakin", "IoT23"],
    "test_datasets": ["UNSW"]
}

# OPCIÓN PERSONALIZADA: Define tu propia combinación
# CROSS_VAL_CONFIG = {
#     "strategy": "Custom",
#     "train_datasets": ["UNSW"],  # Lista de datasets para entrenar
#     "test_datasets": ["Deakin", "IoT23"]  # Lista de datasets para testear
# }

print(f" Estrategia seleccionada: {CROSS_VAL_CONFIG['strategy']}")
print(f" Datasets de entrenamiento: {', '.join(CROSS_VAL_CONFIG['train_datasets'])}")
print(f" Datasets de prueba: {', '.join(CROSS_VAL_CONFIG['test_datasets'])}")
print("\n Configuración lista. Ejecuta la siguiente celda para cargar los datos.")

## 3.2 Carga de Datos para Validación Cruzada

In [ ]:
# ===============================================================
# CARGA DE DATOS SEPARADOS (TRAIN vs TEST)
# ===============================================================

def load_datasets_for_cross_validation(config):
    """
    Carga datasets por separado según configuración de validación cruzada.
    
    Args:
        config: Diccionario con train_datasets y test_datasets
    
    Returns:
        X_train_cv, y_train_cv, X_test_cv, y_test_cv
    """
    print("="*70)
    print(" CARGA DE DATOS PARA VALIDACIÓN CRUZADA")
    print("="*70)
    
    # Guardar configuración original
    original_config = DATASET_ENABLED.copy()
    
    # Habilitar solo datasets de entrenamiento
    print(f"\n Cargando datasets de ENTRENAMIENTO: {config['train_datasets']}")
    for dataset in DATASET_ENABLED.keys():
        DATASET_ENABLED[dataset] = (dataset in config["train_datasets"])
    
    X_train_cv, y_train_cv = load_datasets(discovery_mode=False, verbose=False)
    
    # Habilitar solo datasets de prueba
    print(f"\n Cargando datasets de PRUEBA: {config['test_datasets']}")
    for dataset in DATASET_ENABLED.keys():
        DATASET_ENABLED[dataset] = (dataset in config["test_datasets"])
    
    X_test_cv, y_test_cv = load_datasets(discovery_mode=False, verbose=False)
    
    # Restaurar configuración original
    for dataset in DATASET_ENABLED.keys():
        DATASET_ENABLED[dataset] = original_config[dataset]
    
    print("\n" + "="*70)
    print(" RESUMEN DE DATOS")
    print("="*70)
    print(f"  Datos de entrenamiento: {len(X_train_cv):,} muestras")
    print(f" Datos de prueba: {len(X_test_cv):,} muestras")
    print(f" Total: {len(X_train_cv) + len(X_test_cv):,} muestras")
    
    return X_train_cv, y_train_cv, X_test_cv, y_test_cv

# Cargar datos según configuración
X_train_cv, y_train_cv, X_test_cv, y_test_cv = load_datasets_for_cross_validation(CROSS_VAL_CONFIG)

## 3.3 Preprocesamiento para Validación Cruzada

In [ ]:
# ===============================================================
# PREPROCESAMIENTO PARA VALIDACIÓN CRUZADA
# ===============================================================

print("="*70)
print(" PREPROCESAMIENTO DE DATOS")
print("="*70)

# Label Encoder
le_cv = LabelEncoder()
y_train_cv_encoded = le_cv.fit_transform(y_train_cv)
y_test_cv_encoded = le_cv.transform(y_test_cv)

print(f"\n Clases detectadas: {list(le_cv.classes_)}")
print(f" Número de clases: {len(le_cv.classes_)}")

# Normalización
print("\n Normalizando datos...")
X_train_cv_norm = X_train_cv.astype('float32') / 255.0
X_test_cv_norm = X_test_cv.astype('float32') / 255.0

# Añadir dimensión para CNN
X_train_cv_norm = np.expand_dims(X_train_cv_norm, axis=-1)
X_test_cv_norm = np.expand_dims(X_test_cv_norm, axis=-1)

# Distribución de clases
print("\n DISTRIBUCIÓN DE CLASES:")
print("\n  TRAIN:")
train_counts = pd.Series(y_train_cv).value_counts().sort_index()
for label, count in train_counts.items():
    percentage = (count / len(y_train_cv)) * 100
    print(f"  {label:12s}: {count:6,} muestras ({percentage:5.2f}%)")

print("\n TEST:")
test_counts = pd.Series(y_test_cv).value_counts().sort_index()
for label, count in test_counts.items():
    percentage = (count / len(y_test_cv)) * 100
    print(f"  {label:12s}: {count:6,} muestras ({percentage:5.2f}%)")

print("\n Preprocesamiento completado")
print(f" Shape train: {X_train_cv_norm.shape}")
print(f" Shape test: {X_test_cv_norm.shape}")

## 3.4 Entrenamiento del Modelo (Validación Cruzada)

### Configuración de Early Stopping

Ajusta cuándo detener el entrenamiento automáticamente.

In [ ]:
# ===============================================================
# CONFIGURACIÓN DE EARLY STOPPING
# ===============================================================

#  OPCIÓN 1: Detención rápida (recomendado para pruebas rápidas)
EARLY_STOP_CONFIG = {
    "monitor": "val_loss",      # Métrica a vigilar
    "patience": 3,              # Esperar 3 épocas sin mejora
    "min_delta": 0.10          # Mejora mínima considerada significativa
}

#  OPCIÓN 2: Detención estándar (ACTIVA POR DEFECTO)
# EARLY_STOP_CONFIG = {
#     "monitor": "val_loss",
#     "patience": 15,            # Esperar 15 épocas sin mejora
#     "min_delta": 0.0001
# }

#  OPCIÓN 3: Detención paciente (para asegurar convergencia total)
# EARLY_STOP_CONFIG = {
#     "monitor": "val_loss",
#     "patience": 25,            # Esperar 25 épocas sin mejora
#     "min_delta": 0.0001
# }

#  OPCIÓN 4: Por accuracy en lugar de loss
# EARLY_STOP_CONFIG = {
#     "monitor": "val_accuracy",  # Vigilar accuracy
#     "patience": 10,
#     "min_delta": 0.001,
#     "mode": "max"               # Maximizar accuracy (no minimizar)
# }

#  OPCIÓN 5: Límite de épocas fijo (sin early stopping)
MAX_EPOCHS = 100  # Número máximo de épocas a entrenar

print("  CONFIGURACIÓN DE EARLY STOPPING")
print("="*70)
print(f" Monitor: {EARLY_STOP_CONFIG['monitor']}")
print(f" Patience: {EARLY_STOP_CONFIG['patience']} épocas")
print(f" Min delta: {EARLY_STOP_CONFIG['min_delta']}")
print(f" Max épocas: {MAX_EPOCHS}")
print("\n El entrenamiento se detendrá si:")
print(f"   • No mejora el {EARLY_STOP_CONFIG['monitor']} durante {EARLY_STOP_CONFIG['patience']} épocas consecutivas")
print(f"   • O alcanza {MAX_EPOCHS} épocas")
print("="*70)

In [ ]:
# ===============================================================
# ENTRENAMIENTO CON VALIDACIÓN CRUZADA
# ===============================================================

print("="*70)
print(" ENTRENAMIENTO CON VALIDACIÓN CRUZADA")
print("="*70)
print(f"\n Estrategia: {CROSS_VAL_CONFIG['strategy']}")
print(f"  Train: {', '.join(CROSS_VAL_CONFIG['train_datasets'])}")
print(f" Test: {', '.join(CROSS_VAL_CONFIG['test_datasets'])}")

# Separar datos de entrenamiento en train/val (80/20 del conjunto de entrenamiento)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_cv_norm, y_train_cv_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_train_cv_encoded
)

print(f"\n División de datos:")
print(f"  Train: {len(X_train_split):,} muestras")
print(f"  Val:   {len(X_val_split):,} muestras")
print(f"  Test:  {len(X_test_cv_norm):,} muestras (dataset completo separado)")

# Crear modelo
num_classes_cv = len(le_cv.classes_)
model_cv = build_cnn_model(input_shape=(MAX_LEN, 1), num_classes=num_classes_cv)

print(f"\n  Modelo creado con {num_classes_cv} clases")

# Compilar modelo (NECESARIO cada vez que se crea un modelo nuevo)
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model_cv.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print(" Modelo compilado")

# Configurar callbacks
print("\n Configurando callbacks...")

# 1. EarlyStopping
early_stop_cv = EarlyStopping(
    monitor=EARLY_STOP_CONFIG['monitor'],
    patience=EARLY_STOP_CONFIG['patience'],
    min_delta=EARLY_STOP_CONFIG.get('min_delta', 0.0001),
    mode=EARLY_STOP_CONFIG.get('mode', 'auto'),
    restore_best_weights=True,
    verbose=1
)
print(f"    Early Stopping: monitor='{EARLY_STOP_CONFIG['monitor']}', patience={EARLY_STOP_CONFIG['patience']}")

# 2. ModelCheckpoint - GUARDAR MEJOR MODELO
models_dir = os.path.join(DATA_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
checkpoint_path = os.path.join(models_dir, "best_model_cv.keras")

model_checkpoint_cv = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1,
    mode='max'
)
print(f"    ModelCheckpoint: {checkpoint_path}")

# Entrenar
print("\n Iniciando entrenamiento...")
start_time = time.time()

history_cv = model_cv.fit(
    X_train_split, y_train_split,
    validation_data=(X_val_split, y_val_split),
    epochs=MAX_EPOCHS,
    batch_size=32,
    callbacks=[early_stop_cv, model_checkpoint_cv],  #  Añadido ModelCheckpoint
    verbose=1
)

training_time = time.time() - start_time
print(f"\n  Tiempo de entrenamiento: {training_time/60:.2f} minutos")
print(f" Mejor modelo guardado en: {checkpoint_path}")
print(" Entrenamiento completado")

## 3.5 Evaluación en Dataset de Prueba (Out-of-Domain)

In [ ]:
# ===============================================================
# EVALUACIÓN EN DATASET DE PRUEBA (OUT-OF-DOMAIN)
# ===============================================================

print("="*70)
print(" EVALUACIÓN EN DATASET DE PRUEBA")
print("="*70)
print(f"\n Testeando en: {', '.join(CROSS_VAL_CONFIG['test_datasets'])}")
print(f" Total muestras de prueba: {len(X_test_cv_norm):,}\n")

# Predicciones
y_pred_cv = model_cv.predict(X_test_cv_norm, verbose=0)
y_pred_classes_cv = np.argmax(y_pred_cv, axis=1)

# Métricas
test_loss_cv, test_acc_cv = model_cv.evaluate(X_test_cv_norm, y_test_cv_encoded, verbose=0)
f1_macro_cv = f1_score(y_test_cv_encoded, y_pred_classes_cv, average='macro')
f1_weighted_cv = f1_score(y_test_cv_encoded, y_pred_classes_cv, average='weighted')

print(" MÉTRICAS GLOBALES (OUT-OF-DOMAIN):")
print(f"  Accuracy:     {test_acc_cv*100:.2f}%")
print(f"  F1-Macro:     {f1_macro_cv*100:.2f}%")
print(f"  F1-Weighted:  {f1_weighted_cv*100:.2f}%")
print(f"  Loss:         {test_loss_cv:.4f}")

# Reporte por clase
print("\n" + "="*70)
print(" REPORTE DE CLASIFICACIÓN (OUT-OF-DOMAIN)")
print("="*70)
print(classification_report(
    y_test_cv_encoded, 
    y_pred_classes_cv, 
    target_names=le_cv.classes_,
    digits=4
))

# Matriz de confusión
cm_cv = confusion_matrix(y_test_cv_encoded, y_pred_classes_cv)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_cv, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=le_cv.classes_,
    yticklabels=le_cv.classes_
)
plt.title(f'Matriz de Confusión - {CROSS_VAL_CONFIG["strategy"]}\n(Out-of-Domain Testing)', 
          fontsize=14, fontweight='bold')
plt.ylabel('Etiqueta Real', fontsize=12)
plt.xlabel('Predicción', fontsize=12)
plt.tight_layout()
plt.show()

# Comparar con validación interna
print("\n" + "="*70)
print(" COMPARACIÓN: IN-DOMAIN vs OUT-OF-DOMAIN")
print("="*70)

# Evaluar en validation split (in-domain)
val_loss, val_acc = model_cv.evaluate(X_val_split, y_val_split, verbose=0)
y_pred_val = model_cv.predict(X_val_split, verbose=0)
y_pred_val_classes = np.argmax(y_pred_val, axis=1)
f1_val = f1_score(y_val_split, y_pred_val_classes, average='macro')

print(f"\n IN-DOMAIN (Validation set from training data):")
print(f"  Accuracy:  {val_acc*100:.2f}%")
print(f"  F1-Macro:  {f1_val*100:.2f}%")

print(f"\n OUT-OF-DOMAIN (Completely separate test dataset):")
print(f"  Accuracy:  {test_acc_cv*100:.2f}%")
print(f"  F1-Macro:  {f1_macro_cv*100:.2f}%")

# Calcular diferencia
acc_drop = (val_acc - test_acc_cv) * 100
f1_drop = (f1_val - f1_macro_cv) * 100

print(f"\n DEGRADACIÓN DE RENDIMIENTO:")
print(f"  Accuracy:  {acc_drop:+.2f}%")
print(f"  F1-Macro:  {f1_drop:+.2f}%")

if acc_drop < 5:
    print("\n ¡Excelente generalización! (< 5% degradación)")
elif acc_drop < 10:
    print("\n✓ Buena generalización (5-10% degradación)")
else:
    print("\n  Posible overfitting (> 10% degradación)")